In [28]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/evaluation/benchmark_runner.py
"""
Orchestratore del benchmark RAGAS (cap. sperimentazione).

Due fasi separate, entrambe riprendibili (resume) perche' LLM locale e giudice API sono lenti/limitati:
  1) generate(run_id)        -> usa PipelineHandler (retrieval, reranking, risposta LLM) per ogni query x configurazione
                                e salva contesti, risposta, reference in  <out_dir>/<run_id>_generation.jsonl
  2) evaluate_ragas(run_id)  -> calcola le metriche RAGAS con un giudice LLM esterno, una configurazione alla volta
  3) summary / paired_vs_baseline -> tabelle per la tesi (medie per categoria, differenze con IC bootstrap, win/tie/loss)

run_id serve a distinguere i passaggi: es. "pre_edit" (sidecar senza ritocchi) e "post_edit" (dopo le modifiche manuali).
A parita' di query le configurazioni usano lo STESSO numero di chunk nel contesto (top_j).
"""
import json
import logging
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd

logger = logging.getLogger(__name__)

# Ablation: C0 baseline vettoriale, C1 solo tag, C2 solo distanze, C3 distanze + tag
DEFAULT_CONFIGS: List[Dict[str, Any]] = [
    {"name": "C0_baseline", "mode": "vector", "auto_tags": False},
    {"name": "C1_tags", "mode": "rerank", "order_by": ["TAGS"], "auto_tags": False, "manual_tags": True, "conv_tags": True},
    {"name": "C2_dist", "mode": "rerank", "order_by": ["DIST"], "auto_tags": False, "manual_tags": True, "conv_tags": True},
    {"name": "C3_dist_tags", "mode": "rerank", "order_by": ["DIST", "TAGS"], "auto_tags": False, "manual_tags": True, "conv_tags": True},
    {"name": "C4_dist_tags_joinconv", "mode": "rerank", "order_by": ["DIST", "TAGS", "JOIN_CONV_TAG"], "auto_tags": False, "manual_tags": True, "conv_tags": True},
    {"name": "C5_dist_tags_joinall", "mode": "rerank", "order_by": ["DIST", "TAGS", "JOIN_ALL"], "auto_tags": False, "manual_tags": True, "conv_tags": True},
]

ENGLISH_PROMPT = (
    "You are a research assistant. Answer the question using ONLY the context below. "
    "If the context is insufficient or only partial, say so clearly without inventing facts.\n\n"
    "Question: {q}\n\nContext ({n} chunks):\n{ctx}\n\nAnswer:"
)

BASE_COLS = {"user_input", "retrieved_contexts", "response", "reference", "reference_contexts"}


def _rec_info(rec: Any, idx: int):
    """(chunk_id, testo) da uno ScoredPoint Qdrant o da un dict del reranker (stessa logica di PipelineHandler)."""
    if isinstance(rec, dict):
        payload, cid = rec.get("payload", rec), rec.get("chunk_id")
    else:
        payload, cid = getattr(rec, "payload", {}) or {}, None
    meta = payload.get("metadata") if isinstance(payload.get("metadata"), dict) else payload
    cid = cid or payload.get("chunk_id") or meta.get("chunk_id") or f"chunk_{idx}"
    text = (payload.get("text") or meta.get("text") or payload.get("page_content")
            or meta.get("page_content") or "")
    return cid, text.strip()


def _bootstrap_ci(diffs: np.ndarray, n_boot: int = 5000, seed: int = 0):
    rng = np.random.default_rng(seed)
    means = rng.choice(diffs, size=(n_boot, len(diffs)), replace=True).mean(axis=1)
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def _extract_conv_tags(q: Dict[str, Any]) -> List[str]:
    """Estrae i tag conversazionali dalla voce 'expected_tag' (o 'expected_tags')."""
    raw = q.get("expected_tag") if "expected_tag" in q else q.get("expected_tags", [])
    if isinstance(raw, str):
        return [raw] if raw.strip() else []
    if isinstance(raw, list):
        return list(raw)
    return []


def _extract_manual_tags(q: Dict[str, Any]) -> List[str]:
    """Estrae i tag manuali dalla voce 'tags'."""
    raw = q.get("tags", [])
    if isinstance(raw, str):
        return [raw] if raw.strip() else []
    if isinstance(raw, list):
        return list(raw)
    return []


class RagasBenchmark:
    def __init__(
        self,
        handler,                      # PipelineHandler gia' inizializzato (sidecar + queries_path)
        predetermined_path: str,      # queries_predetermined.json
        normal_path: str,             # queries_normal.json
        out_dir: str,
        configs: Optional[List[Dict[str, Any]]] = None,
        top_j: int = 5,               # chunk nel contesto, uguale per tutte le configurazioni
        top_k: int = 10,              # candidati del retrieval vettoriale
        top_n: int = 7,
        gold_path: Optional[str] = None,   # opzionale: {query_id: [chunk_id, ...]} per le metriche di retrieval
        use_handler_prompt: bool = False,  # True = prompt italiano di handler.generate_rag_response
    ):
        self.h = handler
        self.configs = configs or DEFAULT_CONFIGS
        self.top_j, self.top_k, self.top_n = top_j, top_k, top_n
        self.out_dir = Path(out_dir).expanduser()
        self.out_dir.mkdir(parents=True, exist_ok=True)
        self.use_handler_prompt = use_handler_prompt

        pred = json.load(open(Path(predetermined_path).expanduser(), encoding="utf-8"))
        norm = json.load(open(Path(normal_path).expanduser(), encoding="utf-8"))
        self.queries: List[Dict[str, Any]] = (
            [{**q, "is_predetermined": True} for q in pred] + [{**q, "is_predetermined": False} for q in norm]
        )
        self.gold = json.load(open(Path(gold_path).expanduser(), encoding="utf-8")) if gold_path else {}
        self.pred_path = predetermined_path
        self.norm_path = normal_path

    # ------------------------------------------------------------------ setup
    def register_predetermined(self, which: str = "pred") -> List[str]:
        """Importa query nel sidecar come predeterminate (id con prefisso predq_).
        which: "pred" (21 predeterminate) | "normal" (le altre 21) | "all" (tutte e 42)."""
        ids = []
        if which in ("pred", "all"):
            ids += self.h.add_predetermined_queries_from_file(self.pred_path)
        if which in ("normal", "all"):
            ids += self.h.add_predetermined_queries_from_file(self.norm_path)
        return ids

    def _registered(self) -> set:
        return set(self.h.sidecar_manager.load_data().get("predetermined_queries", {}))

    def edit_status(self) -> pd.DataFrame:
        """Per ogni query: registrata come predeterminata? quante modifiche query<->chunk e tag grafici ha nel sidecar."""
        pq = self.h.sidecar_manager.load_data().get("predetermined_queries", {})
        rows = []
        for q in self.queries:
            e = pq.get(f"predq_{q['id']}")
            rows.append({"query_id": q["id"], "registered": e is not None,
                         "n_dist_edits": len((e or {}).get("adjacencies", {})),
                         "n_graph_tags": len((e or {}).get("graphically_assigned_tags", []))})
        return pd.DataFrame(rows)

    def prepare(
        self,
        query_id: str,
        order_by: Optional[List[str]] = None,
        auto_tags: bool = False,
        manual_tags: bool = True,
        conv_tags: bool = True,
    ) -> Dict[str, Any]:
        """Porta l'handler sulla query (retrieval + reranking): poi si usa il widget come in handler.update_widget_view(...)."""
        q = next(q for q in self.queries if q["id"] == query_id)
        h = self.h
        h.set_parameters(top_j=self.top_j, top_k=self.top_k, top_n=self.top_n, order_by=order_by or ["DIST", "TAGS"])
        h.update_query(new_text=q["query"])

        conv_list = _extract_conv_tags(q) if conv_tags else []
        ass_list = _extract_manual_tags(q) if manual_tags else []

        h.update_query_tags(new_conv_list=conv_list, new_ass_list=ass_list, auto_ass_flag=auto_tags, print_debug=True)
        h.run_vectorial_retrieval()
        h.run_top_j_distance_tag_reranking()
        return q

    def check_similarity(self, warn_above: float = 0.80) -> pd.DataFrame:
        """Per ogni normale, la massima similarita' coseno con le predeterminate (>= soglia handler = collasso)."""
        pred = [q for q in self.queries if q["is_predetermined"]]
        norm = [q for q in self.queries if not q["is_predetermined"]]
        e = self.h.embeddings
        P = np.array(e.embed_documents([q["query"] for q in pred]))
        N = np.array(e.embed_documents([q["query"] for q in norm]))
        P /= np.linalg.norm(P, axis=1, keepdims=True)
        N /= np.linalg.norm(N, axis=1, keepdims=True)
        S = N @ P.T
        pred_ids = [p["id"] for p in pred]
        rows = []
        for i, q in enumerate(norm):
            row = {"normal_id": q["id"], "max_sim": float(S[i].max()), "closest_pred": pred[int(S[i].argmax())]["id"]}
            pw = q.get("paired_with")
            if pw in pred_ids:
                row["paired_with"] = pw
                row["sim_with_partner"] = float(S[i][pred_ids.index(pw)])
            rows.append(row)
        df = pd.DataFrame(rows).sort_values("max_sim", ascending=False)
        bad = df[df.max_sim >= warn_above]
        if len(bad):
            logger.warning(f"!> {len(bad)} query normali troppo simili a una predeterminata: {bad.normal_id.tolist()}")
        return df

    # ------------------------------------------------------------- generation
    def _gen_path(self, run_id: str) -> Path:
        return self.out_dir / f"{run_id}_generation.jsonl"

    def load_generation(self, run_id: str) -> pd.DataFrame:
        p = self._gen_path(run_id)
        return pd.read_json(p, lines=True) if p.exists() else pd.DataFrame()

    def _answer(self, query: str, records: List[Any]) -> str:
        if self.use_handler_prompt:
            return self.h.generate_rag_response(records=records)["response"]
        ctx = "\n\n".join(f"[{i}] {_rec_info(r, i)[1]}" for i, r in enumerate(records, 1))
        out = self.h.llm.invoke(ENGLISH_PROMPT.format(q=query, n=len(records), ctx=ctx))
        return (out.content if hasattr(out, "content") else str(out)).strip()

    def _retrieval_metrics(self, qid: str, ids: List[str]) -> Dict[str, Any]:
        gold = set(self.gold.get(qid, []))
        if not gold:
            return {}
        hits = [i for i, c in enumerate(ids) if c in gold]
        return {"gold_hit": int(bool(hits)),
                "gold_recall": len(gold & set(ids)) / len(gold),
                "gold_precision": len(gold & set(ids)) / max(len(ids), 1),
                "gold_mrr": 1.0 / (hits[0] + 1) if hits else 0.0}

    def generate(self, run_id: str, query_ids: Optional[List[str]] = None, with_answers: bool = True) -> pd.DataFrame:
        """Esegue query x configurazioni. Riprende da dove era arrivato (chiave: query_id + config)."""
        path = self._gen_path(run_id)
        done = set()
        if path.exists():
            done = {(r["query_id"], r["config"]) for r in map(json.loads, open(path, encoding="utf-8"))}
        qs = [q for q in self.queries if query_ids is None or q["id"] in query_ids]
        registered = self._registered()

        with open(path, "a", encoding="utf-8") as f:
            for n, q in enumerate(qs, 1):
                todo = [c for c in self.configs if (q["id"], c["name"]) not in done]
                if not todo:
                    continue
                h = self.h
                h.set_parameters(top_j=self.top_j, top_k=self.top_k, top_n=self.top_n)
                h.update_query(new_text=q["query"])
                predq_ok = bool(h.is_predq) == (f"predq_{q['id']}" in registered)
                if not predq_ok:
                    logger.warning(f"!> {q['id']}: riconoscimento predetermined inatteso (is_predq={h.is_predq})")
                h.run_vectorial_retrieval()

                for c in todo:
                    t0 = time.perf_counter()
                    if c["mode"] == "vector":
                        records = list(h.raw_records)[: self.top_j]
                    else:
                        h.set_parameters(order_by=c.get("order_by", []))

                        conv_list = _extract_conv_tags(q) if c.get("conv_tags", False) else []
                        ass_list = _extract_manual_tags(q) if c.get("manual_tags", False) else []

                        h.update_query_tags(new_conv_list=conv_list, new_ass_list=ass_list,
                                            auto_ass_flag=c.get("auto_tags", False))
                        h.run_top_j_distance_tag_reranking()
                        records = list(h.top_j_reranked)
                    info = [_rec_info(r, i) for i, r in enumerate(records)]
                    ids = [i for i, _ in info]
                    row = {
                        "run_id": run_id, "query_id": q["id"], "category": q["category"],
                        "subtype": q.get("subtype"), "is_predetermined": q["is_predetermined"],
                        "predq_ok": predq_ok, "registered": f"predq_{q['id']}" in registered, "config": c["name"],
                        "user_input": q["query"], "reference": q["reference"],
                        "retrieved_ids": ids, "retrieved_contexts": [t for _, t in info],
                        "response": self._answer(q["query"], records) if with_answers else "",
                        "seconds": round(time.perf_counter() - t0, 2),
                        **self._retrieval_metrics(q["id"], ids),
                    }
                    f.write(json.dumps(row, ensure_ascii=False) + "\n")
                    f.flush()
                logger.info(f"?> [{n}/{len(qs)}] {q['id']} completata ({len(todo)} config)")
        return self.load_generation(run_id)

    # ------------------------------------------------------------------ RAGAS
    def evaluate_ragas(self, run_id: str, judge_llm, max_workers: int = 2, timeout: int = 300,
                       metrics: Optional[List[Any]] = None, configs: Optional[List[str]] = None,
                       max_nan_frac: Optional[float] = 0.3) -> pd.DataFrame:
        """
        judge_llm: modello chat LangChain del giudice (es. ChatGoogleGenerativeAI / ChatGroq / ChatOpenAI).
        Embedding per ResponseRelevancy: quelli locali dell'handler (nessun costo API).
        Salva un CSV per configurazione: se esiste gia' viene saltata (riprendibile).
        max_nan_frac: se per una metrica la quota di NaN (fallimenti del giudice) supera questa soglia, il CSV NON
                      viene salvato col nome normale (cosi' alla ripartenza la config viene rifatta): finisce in
                      <run_id>_<config>_REJECTED.csv e la valutazione si ferma con un errore. None = nessun controllo.
        """
        from ragas import EvaluationDataset, evaluate
        from ragas.embeddings import LangchainEmbeddingsWrapper
        from ragas.llms import LangchainLLMWrapper
        from ragas.metrics import (Faithfulness, LLMContextPrecisionWithReference,
                                   LLMContextRecall, ResponseRelevancy)
        from ragas.run_config import RunConfig

        gen = self.load_generation(run_id)
        if gen.empty:
            raise RuntimeError("Nessuna generazione trovata: esegui prima generate().")
        llm = LangchainLLMWrapper(judge_llm)

        # Ragas fa getattr(embeddings, "model") sull'oggetto interno per il tracking: su FastEmbedEmbeddings .model e'
        # l'oggetto fastembed.TextEmbedding (non una str) -> ValidationError Pydantic. NON sovrascrivere .model
        # (e' il motore di embedding dell'handler): a Ragas si passa un adattatore che non espone .model.
        from langchain_core.embeddings import Embeddings

        class _NoModelAttr(Embeddings):
            def __init__(self, inner):
                self._inner = inner

            def embed_documents(self, texts):
                return self._inner.embed_documents(texts)

            def embed_query(self, text):
                return self._inner.embed_query(text)

        emb = LangchainEmbeddingsWrapper(_NoModelAttr(self.h.embeddings))
        metrics = metrics or [Faithfulness(), ResponseRelevancy(),
                              LLMContextPrecisionWithReference(), LLMContextRecall()]

        for cname in (configs or [c["name"] for c in self.configs]):
            out = self.out_dir / f"{run_id}_{cname}_ragas.csv"
            if out.exists():
                logger.info(f"?> {cname}: gia' valutata, salto")
                continue
            sub = gen[gen.config == cname].reset_index(drop=True)
            ds = EvaluationDataset.from_list(sub[["user_input", "retrieved_contexts", "response", "reference"]]
                                             .to_dict("records"))
            # Isola Ragas dal loop di Jupyter tramite un Thread isolato permettendone l'esecuzione senza problemi di contesti
            import concurrent.futures
            
            def run_eval():
                return evaluate(dataset=ds, metrics=metrics, llm=llm, embeddings=emb,
                                run_config=RunConfig(max_workers=max_workers, timeout=timeout),
                                raise_exceptions=False, show_progress=True)

            with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
                res = executor.submit(run_eval).result()
                
            df = res.to_pandas()
            mcols = [c for c in df.columns if c not in BASE_COLS]
            if len(df) != len(sub):
                raise RuntimeError(f"{cname}: Ragas ha restituito {len(df)} righe, attese {len(sub)}: nulla salvato.")
            df = pd.concat([sub[["run_id", "query_id", "category", "subtype", "is_predetermined", "config"]],
                            df[mcols]], axis=1)

            # Controllo NaN: timeout, JSON non valido o quota API esaurita diventano NaN (raise_exceptions=False).
            nan_frac = df[mcols].isna().mean()
            if max_nan_frac is not None and (nan_frac > max_nan_frac).any():
                rej = self.out_dir / f"{run_id}_{cname}_REJECTED.csv"
                df.to_csv(rej, index=False)
                raise RuntimeError(
                    f"{cname}: troppi NaN, CSV NON salvato come valutato (copia in {rej.name}). "
                    f"Quota NaN per metrica: {nan_frac.round(2).to_dict()} (soglia {max_nan_frac}). "
                    f"Rilancia la cella per rifare questa config, oppure alza max_nan_frac / usa None per accettarli.")

            # Scrittura atomica: un'interruzione durante il salvataggio non lascia un CSV a meta' che verrebbe saltato.
            tmp = out.with_name(out.name + ".tmp")
            df.to_csv(tmp, index=False)
            tmp.replace(out)
            logger.info(f"?> {cname}: salvato {out.name} (NaN: {int(df[mcols].isna().sum().sum())})")
        return self.load_ragas(run_id)

    def load_ragas(self, run_id: str) -> pd.DataFrame:
        fs = sorted(self.out_dir.glob(f"{run_id}_*_ragas.csv"))
        return pd.concat([pd.read_csv(f) for f in fs], ignore_index=True) if fs else pd.DataFrame()

    # ---------------------------------------------------------------- reports
    @staticmethod
    def _metric_cols(df: pd.DataFrame) -> List[str]:
        meta = {"run_id", "query_id", "category", "subtype", "is_predetermined", "config"}
        return [c for c in df.columns if c not in meta and pd.api.types.is_numeric_dtype(df[c])]

    def summary(self, df: pd.DataFrame, by: str = "category") -> pd.DataFrame:
        """Media per configurazione x (category | is_predetermined | subtype), con n e NaN ignorati."""
        return df.groupby(["config", by])[self._metric_cols(df)].mean().round(3)

    def paired_vs_baseline(self, df: pd.DataFrame, baseline: str = "C0_baseline", tol: float = 0.01,
                           subset: Optional[pd.Series] = None) -> pd.DataFrame:
        """Differenza media appaiata (config - baseline) con IC95% bootstrap e win/tie/loss per query."""
        d = df if subset is None else df[subset]
        rows = []
        for m in self._metric_cols(d):
            piv = d.pivot_table(index="query_id", columns="config", values=m, aggfunc="first")
            for c in [c for c in piv.columns if c != baseline]:
                pair = piv[[baseline, c]].dropna()
                if len(pair) < 3:
                    continue
                diff = (pair[c] - pair[baseline]).to_numpy()
                lo, hi = _bootstrap_ci(diff)
                rows.append({"metric": m, "config": c, "n": len(pair), "n_nan": int(piv[c].isna().sum()),
                             "mean_diff": round(float(diff.mean()), 3), "ci95_low": round(lo, 3),
                             "ci95_high": round(hi, 3), "win": int((diff > tol).sum()),
                             "tie": int((np.abs(diff) <= tol).sum()), "loss": int((diff < -tol).sum())})
        return pd.DataFrame(rows)

    def retrieval_summary(self, run_id: str) -> pd.DataFrame:
        """Metriche di retrieval su gold chunk (nessun LLM): hit, recall, precision, MRR per configurazione."""
        g = self.load_generation(run_id)
        cols = [c for c in ["gold_hit", "gold_recall", "gold_precision", "gold_mrr"] if c in g.columns]
        return g.groupby(["config", "category"])[cols].mean().round(3) if cols else pd.DataFrame()

EOF

## COLOSSEUM DATASET

In [ ]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/evaluation/build_colosseum_corpus.py
#!/usr/bin/env python3
"""
build_colosseum_corpus.py - costruisce il corpus "Colosseo" (chunk + tag provvisori) da Wikipedia e da testi locali.

Uso (sull'hub o su Colab: serve Internet):
    python build_colosseum_corpus.py --out ~/tesi_graphrag/data/queries/colosseum \
        --papers ~/tesi_graphrag/data/raw/colosseum_papers --contact "tuo_nome / universita"

Produce in --out:
    chunks_colosseum.json   formato atteso da json_ingest.ingest_chunks_json
    chunks_preview.tsv      revisione veloce: id, tag, parole, sezione, inizio del testo
    sources_colosseum.json  provenienza: URL, revisione Wikipedia, data di scarico, licenza
    corpus_report.txt       chunk per documento e copertura per tag
    raw/<doc_id>.txt        testo grezzo scaricato (cache: i rilanci non riscaricano; --refetch per forzare)

Revisione dei tag: i tag sono PROVVISORI (regole su titolo di sezione, vedi RULES). Dopo la revisione si possono fissare
chunk per chunk con --overrides tag_overrides.json  ({"chunk_id": ["Tag1", "Tag2"], ...}).

Licenze: il testo di Wikipedia e' CC BY-SA 4.0 (citare titolo, revisione e data: sono in sources_colosseum.json).
Gli altri testi vanno citati in tesi; la cartella data/ e i file .json/.pdf/.txt sono gia' esclusi da .gitignore.
"""
import argparse
import json
import re
import statistics
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

# --------------------------------------------------------------------------- configurazione
TAGS = ["History", "Architecture", "Spectacles", "Archaeology", "Conservation", "Culture"]

# (titolo Wikipedia, doc_id)
WIKI_ARTICLES = [
    ("Colosseum", "wiki_colosseum"),
    ("Domus Aurea", "wiki_domus_aurea"),
    ("Colossus of Nero", "wiki_colossus_of_nero"),
    ("Ludus Magnus", "wiki_ludus_magnus"),
    ("Venatio", "wiki_venatio"),
    ("Naumachia", "wiki_naumachia"),
    ("Velarium", "wiki_velarium"),
    ("Arch of Constantine", "wiki_arch_of_constantine"),
]

# testi locali (PDF leggibili senza OCR, oppure .txt/.md): nome file -> parametri. Adatta i nomi ai tuoi file.
LOCAL_DOCS = {
    "cerone_arena_floor.pdf": {"doc_id": "paper_arena_floor", "tags": ["Archaeology", "Conservation"],
                               "title": "Cerone, Viskovic, Fumagalli, Rea - The Re-formation of Arena Floor Deck in the Colosseum in Roma",
                               "skip_first_words": 40, "max_chunks": 10},
    "firerisk_post_earthquake_fire.txt": {"doc_id": "web_post_earthquake_fire", "tags": ["Conservation", "Architecture"],
                                          "title": "FireRiskHeritage.net - Did Rome's Colosseum Suffer a Post-Earthquake Fire in 271 A.D.?",
                                          "skip_first_words": 0, "max_chunks": 6},
    "colosseum_net_underground.txt": {"doc_id": "web_underground", "tags": ["Archaeology", "Architecture"],
                                      "title": "the-colosseum.net - Underground (the hypogea)",
                                      "skip_first_words": 0, "max_chunks": 8},
    "mordanova_defelice_2020.pdf": {"doc_id": "paper_seismic_dem", "tags": ["Conservation", "Architecture"],
                                    "title": "Mordanova, de Felice - Seismic Assessment of Archaeological Heritage Using Discrete Element Method",
                                    "skip_first_words": 80, "max_chunks": 10},
}

# regole PROVVISORIE per i tag dei chunk Wikipedia: la prima regola che combacia con il percorso della sezione
# (es. "history > construction") decide; altrimenti vale il tag di default del documento.
DEFAULT_TAGS = {
    "wiki_colosseum": ["History"],
    "wiki_domus_aurea": ["History"],
    "wiki_colossus_of_nero": ["History"],
    "wiki_ludus_magnus": ["Spectacles"],
    "wiki_venatio": ["Spectacles"],
    "wiki_naumachia": ["Spectacles"],
    "wiki_velarium": ["Architecture"],
    "wiki_arch_of_constantine": ["History"],
}
RULES = {
    "wiki_colosseum": [
        (r"hypogeum|underground|arena floor|\barena\b", ["Archaeology", "Architecture"]),
        (r"religio|christian|martyr|church|stations of the cross|pilgrim", ["Culture"]),
        (r"flora|plant|botan|vegetation", ["Conservation"]),
        (r"earthquake|stripp|spoil|restor|preserv|conserv|damage|pollution|ruin", ["Conservation"]),
        (r"gladiat|games|event|show|hunt|execution|naval|naumach|spectacle|venatio|entertain|inaugur", ["Spectacles"]),
        (r"exterior|fa[cç]ade|interior|seating|access|architect|design|structure|material|dimension|velarium|awning", ["Architecture"]),
        (r"cultural|influence|legacy|popular|symbol|death penalty|in culture", ["Culture"]),
    ],
    "wiki_domus_aurea": [
        (r"rediscover|excavat|archaeolog", ["Archaeology"]),
        (r"preserv|restor|collapse|conserv|damage|closure|water", ["Conservation"]),
        (r"architect|design|decorat|room|octagon|painting|fresco|description|layout|plan", ["Architecture"]),
    ],
    "wiki_ludus_magnus": [
        (r"excavat|archaeolog|remains|discover|ruin|rediscover", ["Archaeology"]),
        (r"architect|structure|design|building|cells|training arena|layout|description", ["Architecture"]),
    ],
    "wiki_arch_of_constantine": [
        (r"restor|preserv|conserv", ["Conservation"]),
        (r"architect|design|decorat|relief|spolia|description|sculpt|inscription", ["Architecture"]),
    ],
}

SKIP_SECTIONS = {"see also", "references", "notes", "citations", "bibliography", "further reading", "external links",
                 "sources", "footnotes", "gallery", "works cited", "explanatory notes", "footnotes and references"}
MAX_WORDS = 200        # parole massime per chunk
MIN_WORDS = 25         # sotto questa soglia un chunk si unisce al precedente dello stesso documento
MIN_TAIL = 40          # una coda piu' corta di cosi' si unisce al chunk precedente della stessa sezione

WIKI_API = "https://en.wikipedia.org/w/api.php"
HEAD = re.compile(r"^(={2,6})\s*(.+?)\s*\1\s*$")


# --------------------------------------------------------------------------- Wikipedia
def fetch_wikipedia(title, contact=""):
    """Testo semplice di una voce (con titoli di sezione '== X ==') + revisione e data. Una voce per chiamata."""
    import requests
    ua = f"TesiGraphRAG-corpus-builder/1.0 (university thesis project{'; ' + contact if contact else ''})"
    params = {"action": "query", "format": "json", "formatversion": 2, "redirects": 1, "titles": title,
              "prop": "extracts|revisions", "explaintext": 1, "exsectionformat": "wiki", "rvprop": "ids|timestamp"}
    r = requests.get(WIKI_API, params=params, headers={"User-Agent": ua}, timeout=60)
    r.raise_for_status()
    page = r.json()["query"]["pages"][0]
    if page.get("missing"):
        raise LookupError(f"voce Wikipedia inesistente: {title}")
    rev = page["revisions"][0]
    return {"title": page["title"], "text": page["extract"], "revid": rev["revid"], "timestamp": rev["timestamp"]}


# --------------------------------------------------------------------------- parsing e chunking
def parse_sections(text):
    """-> [(percorso: [titoli], testo)]. Il testo prima del primo titolo ha percorso ['Lead']."""
    sections, buf, stack, cur = [], [], [], ["Lead"]

    def flush():
        t = "\n".join(buf).strip()
        if t:
            sections.append((list(cur), t))

    for line in text.splitlines():
        m = HEAD.match(line.strip())
        if m:
            flush()
            buf.clear()
            level, title = len(m.group(1)), m.group(2).strip()
            while stack and stack[-1][0] >= level:
                stack.pop()
            stack.append((level, title))
            cur = [t for _, t in stack]
        else:
            buf.append(line)
    flush()
    return [(p, t) for p, t in sections if not any(x.lower() in SKIP_SECTIONS for x in p)]


def words(s):
    return len(s.split())


def paragraphs(text):
    out = []
    for p in re.split(r"\n\s*\n", text):
        p = re.sub(r"\s+", " ", p).strip()
        if p:
            out.append(p)
    return out


def split_sentences(par):
    return re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\"“‘(])", par)


def chunk_paragraphs(pars, max_words=MAX_WORDS):
    """Raggruppa paragrafi fino a max_words; un paragrafo piu' lungo viene spezzato sui confini di frase."""
    units = []
    for p in pars:
        if words(p) <= max_words:
            units.append(p)
            continue
        buf, k = [], 0
        for s in split_sentences(p):
            if buf and k + words(s) > max_words:
                units.append(" ".join(buf))
                buf, k = [], 0
            buf.append(s)
            k += words(s)
        if buf:
            units.append(" ".join(buf))
    chunks, cur, n = [], [], 0
    for u in units:
        w = words(u)
        if cur and n + w > max_words:
            chunks.append("\n\n".join(cur))
            cur, n = [], 0
        cur.append(u)
        n += w
    if cur:
        chunks.append("\n\n".join(cur))
    if len(chunks) > 1 and words(chunks[-1]) < MIN_TAIL:       # coda troppo corta: si attacca al chunk precedente
        tail = chunks.pop()
        chunks[-1] = chunks[-1] + "\n\n" + tail
    return chunks


def assign_tags(doc_id, path):
    heading = " > ".join(path).lower()
    for pattern, tags in RULES.get(doc_id, []):
        if re.search(pattern, heading):
            return list(tags)
    return list(DEFAULT_TAGS.get(doc_id, ["History"]))


def build_wiki_chunks(doc_id, text):
    """-> lista di dict {section, tags, text} (senza chunk_id)."""
    out = []
    for path, body in parse_sections(text):
        tags = assign_tags(doc_id, path)
        for ch in chunk_paragraphs(paragraphs(body)):
            if out and words(ch) < MIN_WORDS:                    # frammento minuscolo: si unisce al precedente
                out[-1]["text"] += "\n\n" + ch
            else:
                out.append({"section": " > ".join(path), "tags": tags, "text": ch})
    return out


# --------------------------------------------------------------------------- testi locali (PDF / txt)
def normalize_pdf_text(raw):
    """Ripulisce il testo estratto da un PDF: sillabazione, a capo dentro i paragrafi, riferimenti bibliografici."""
    t = raw.replace("\r", "")
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)                           # parole spezzate a fine riga
    m = re.search(r"\n\s*(references|bibliography|acknowledg(?:e)?ments?)\s*\n", t, flags=re.I)
    if m and m.start() > 0.3 * len(t):                               # taglia i riferimenti (solo se nella parte finale)
        t = t[:m.start()]
    lines = [ln.strip() for ln in t.split("\n")]
    lens = [len(ln) for ln in lines if len(ln) > 20]
    med = statistics.median(lens) if lens else 80
    paras, cur = [], []
    for ln in lines:
        if not ln or re.fullmatch(r"\d{1,4}", ln):                   # riga vuota o solo numero di pagina
            if cur:
                paras.append(" ".join(cur)); cur = []
            continue
        cur.append(ln)
        if re.search(r"[.!?:]$", ln) and len(ln) < 0.65 * med:       # riga corta che chiude una frase: fine paragrafo
            paras.append(" ".join(cur)); cur = []
    if cur:
        paras.append(" ".join(cur))
    return "\n\n".join(paras)


def read_local_text(path):
    p = Path(path)
    if p.suffix.lower() == ".pdf":
        import fitz  # pymupdf (gia' in requirements.txt)
        with fitz.open(p) as doc:
            raw = "\n".join(page.get_text("text") for page in doc)
        return normalize_pdf_text(raw)
    return normalize_pdf_text(p.read_text(encoding="utf-8", errors="ignore"))


def build_local_chunks(text, spec):
    pars = paragraphs(text)
    skip = spec.get("skip_first_words", 0)
    while skip > 0 and pars:                                          # salta titolo/autori/affiliazioni all'inizio
        w = words(pars[0])
        if w > skip and skip < 0.5 * w:
            break
        skip -= w
        pars.pop(0)
    chunks = chunk_paragraphs(pars)
    chunks = [c for c in chunks if words(c) >= MIN_WORDS][: spec.get("max_chunks", 10)]
    return [{"section": "full text", "tags": list(spec["tags"]), "text": c} for c in chunks]


# --------------------------------------------------------------------------- orchestrazione
def validate_tags(tags, where):
    bad = [t for t in tags if t not in TAGS]
    if bad:
        raise ValueError(f"{where}: tag non validi {bad}; validi: {TAGS}")


def assemble(out_dir, papers_dir=None, contact="", refetch=False, overrides_path=None, fetcher=fetch_wikipedia):
    out_dir = Path(out_dir).expanduser()
    raw_dir = out_dir / "raw"
    raw_dir.mkdir(parents=True, exist_ok=True)
    chunks, sources, seen_titles, warnings = [], [], set(), []

    for title, doc_id in WIKI_ARTICLES:
        cache, meta = raw_dir / f"{doc_id}.txt", raw_dir / f"{doc_id}.meta.json"
        if cache.exists() and meta.exists() and not refetch:
            info = json.loads(meta.read_text(encoding="utf-8"))
            text = cache.read_text(encoding="utf-8")
        else:
            try:
                got = fetcher(title, contact)
            except Exception as e:                                    # voce mancante / rete: si prosegue con le altre
                warnings.append(f"{doc_id}: non scaricata ({type(e).__name__}: {e})")
                continue
            text = got["text"]
            info = {"title": got["title"], "revid": got["revid"], "timestamp": got["timestamp"],
                    "retrieved": datetime.now(timezone.utc).strftime("%Y-%m-%d")}
            cache.write_text(text, encoding="utf-8")
            meta.write_text(json.dumps(info, ensure_ascii=False, indent=1), encoding="utf-8")
            time.sleep(1)                                             # cortesia verso l'API
        if info["title"] in seen_titles:                              # un redirect puo' riportare a una voce gia' presa
            warnings.append(f"{doc_id}: '{title}' porta a '{info['title']}' gia' inclusa: saltata")
            continue
        seen_titles.add(info["title"])
        parts = build_wiki_chunks(doc_id, text)
        if not parts:
            warnings.append(f"{doc_id}: nessun chunk prodotto")
            continue
        for i, c in enumerate(parts, 1):
            chunks.append({"doc_id": doc_id, "chunk_index": i, "source_title": info["title"], **c})
        sources.append({"doc_id": doc_id, "title": info["title"], "type": "wikipedia",
                        "url": "https://en.wikipedia.org/wiki/" + info["title"].replace(" ", "_"),
                        "revision_id": info["revid"], "revision_timestamp": info["timestamp"],
                        "retrieved": info["retrieved"], "license": "CC BY-SA 4.0", "n_chunks": len(parts)})

    if papers_dir and Path(papers_dir).expanduser().exists():
        pdir = Path(papers_dir).expanduser()
        for fname, spec in LOCAL_DOCS.items():
            f = pdir / fname
            if not f.exists():
                warnings.append(f"{spec['doc_id']}: file '{fname}' non trovato in {pdir} (saltato)")
                continue
            parts = build_local_chunks(read_local_text(f), spec)
            for i, c in enumerate(parts, 1):
                chunks.append({"doc_id": spec["doc_id"], "chunk_index": i, "source_title": spec["title"], **c})
            sources.append({"doc_id": spec["doc_id"], "title": spec["title"], "type": "local", "file": fname,
                            "license": "da verificare (citare in tesi; non ridistribuire)", "n_chunks": len(parts)})
        known = set(LOCAL_DOCS)
        for f in pdir.iterdir():
            if f.is_file() and f.name not in known:
                warnings.append(f"file ignorato in {pdir}: '{f.name}' (non e' in LOCAL_DOCS)")

    overrides = {}
    if overrides_path:
        overrides = json.loads(Path(overrides_path).expanduser().read_text(encoding="utf-8"))
    for c in chunks:
        c["chunk_id"] = f"{c['doc_id']}_{c['chunk_index']:02d}"
        validate_tags(c["tags"], c["chunk_id"])
    ids = {c["chunk_id"] for c in chunks}
    unknown = [k for k in overrides if k not in ids]
    if unknown:
        raise ValueError(f"overrides per chunk inesistenti: {unknown[:10]}")
    for c in chunks:
        if c["chunk_id"] in overrides:
            validate_tags(overrides[c["chunk_id"]], c["chunk_id"])
            c["tags"] = list(overrides[c["chunk_id"]])
        c["true_domain"] = c["tags"][0]
        c["text"] = c["text"].strip()

    final = [{"chunk_id": c["chunk_id"], "doc_id": c["doc_id"], "true_domain": c["true_domain"],
              "chunk_index": c["chunk_index"], "tags": c["tags"], "text": c["text"],
              "section": c["section"], "source_title": c["source_title"]} for c in chunks]
    (out_dir / "chunks_colosseum.json").write_text(json.dumps(final, ensure_ascii=False, indent=1), encoding="utf-8")
    (out_dir / "sources_colosseum.json").write_text(json.dumps(sources, ensure_ascii=False, indent=1), encoding="utf-8")
    with open(out_dir / "chunks_preview.tsv", "w", encoding="utf-8") as f:
        f.write("chunk_id\ttags\twords\tsection\tstart\n")
        for c in final:
            f.write(f"{c['chunk_id']}\t{'+'.join(c['tags'])}\t{words(c['text'])}\t{c['section']}\t"
                    f"{c['text'][:140].replace(chr(10), ' ')}\n")

    rep = [f"chunk totali: {len(final)}  (obiettivo ~80-110)", "", "per documento:"]
    for s in sources:
        rep.append(f"  {s['doc_id']:28s} {s['n_chunks']:3d}  {s['title']}")
    rep += ["", "copertura per tag (quota di chunk che hanno il tag; obiettivo ~12-30% ciascuno):"]
    for t in TAGS:
        n = sum(t in c["tags"] for c in final)
        rep.append(f"  {t:13s} {n:3d} chunk  {100 * n / max(len(final), 1):4.0f}%")
    wl = [words(c["text"]) for c in final]
    if wl:
        rep += ["", f"parole per chunk: min {min(wl)}, mediana {int(statistics.median(wl))}, max {max(wl)}"]
    if warnings:
        rep += ["", "AVVISI:"] + [f"  - {w}" for w in warnings]
    (out_dir / "corpus_report.txt").write_text("\n".join(rep), encoding="utf-8")
    print("\n".join(rep))
    return final


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--out", required=True, help="cartella di output (es. ~/tesi_graphrag/data/queries/colosseum)")
    ap.add_argument("--papers", default=None, help="cartella con i PDF/txt locali (vedi LOCAL_DOCS)")
    ap.add_argument("--contact", default="", help="contatto da mettere nello User-Agent verso Wikipedia (nome/universita')")
    ap.add_argument("--refetch", action="store_true", help="riscarica Wikipedia anche se in cache")
    ap.add_argument("--overrides", default=None, help="JSON {chunk_id: [tag, ...]} che sostituisce i tag provvisori")
    a = ap.parse_args(argv)
    assemble(a.out, a.papers, a.contact, a.refetch, a.overrides)


if __name__ == "__main__":
    sys.exit(main())

EOF

In [29]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/evaluation/degrade_tags.py
"""degrade_tags - versione "utente medio" dei TAG (il corrispettivo di distance_presets.degrade_plan per i tag).
Parte dai tag dell'esperto (chunks_colosseum.json) e produce una copia con errori realistici, riproducibile col seed:
  - missing_frac : quota di chunk lasciati SENZA tag (l'utente non ha fatto in tempo a taggarli);
  - wrong_frac   : tra i chunk taggati, quota con un tag SBAGLIATO (sostituito con un altro tag a caso);
  - drop_secondary: nei chunk con piu' tag, l'utente tiene di solito solo il primo.
Il piano di distanze si degrada a parte con degrade_plan(); i due si combinano nel run 'post_edit_user'."""
import random
from collections import Counter


def degrade_tags(chunks, all_tags, missing_frac=0.30, wrong_frac=0.15, drop_secondary=True, seed=0):
    rng = random.Random(seed)
    out, log = [], Counter()
    for c in chunks:
        d = dict(c)
        tags = list(c["tags"])
        if rng.random() < missing_frac:
            d["tags"] = []; log["missing"] += 1
        else:
            if drop_secondary and len(tags) > 1:
                tags = tags[:1]; log["secondary_dropped"] += 1
            if tags and rng.random() < wrong_frac:
                i = rng.randrange(len(tags))
                tags[i] = rng.choice([t for t in all_tags if t not in tags]); log["wrong"] += 1
            d["tags"] = tags
        out.append(d)
    return out, dict(log)
EOF

In [26]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/evaluation/finalize_colosseum.py
#!/usr/bin/env python3
"""finalize_colosseum.py - applica la revisione manuale (tag + chunk scartati) a chunks_colosseum.json.
Uso:  python finalize_colosseum.py --chunks data/queries/colosseum/chunks_colosseum.json --review colosseum_review.json
Scrive accanto al file di input: chunks_colosseum_final.json (da ingerire), tag_overrides.json, review_report.txt.
Gli id non cambiano (restano 'buchi' nella numerazione dei chunk scartati). Gli chunk non presenti nella revisione
(es. testi aggiunti dopo) tengono i tag provvisori e sono elencati nel report."""
import argparse, json
from pathlib import Path
from collections import Counter
TAGS = ["History", "Architecture", "Spectacles", "Archaeology", "Conservation", "Culture"]

def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--chunks", required=True); ap.add_argument("--review", required=True)
    a = ap.parse_args()
    cp = Path(a.chunks).expanduser(); out = cp.parent
    chunks = json.load(open(cp, encoding="utf-8")); rev = json.load(open(Path(a.review).expanduser(), encoding="utf-8"))
    drop, tags = set(rev["drop"]), rev["tags"]
    for k, v in tags.items():
        assert v and all(x in TAGS for x in v), f"{k}: tag non validi {v}"
    final, pending = [], []
    for c in chunks:
        if c["chunk_id"] in drop: continue
        c = dict(c)
        if c["chunk_id"] in rev.get("text", {}): c["text"] = rev["text"][c["chunk_id"]]   # pulizia manuale (es. intestazioni/didascalie del PDF)
        if c["chunk_id"] in tags: c["tags"] = list(tags[c["chunk_id"]])
        else: pending.append(c["chunk_id"])
        c["true_domain"] = c["tags"][0]
        final.append(c)
    json.dump(final, open(out / "chunks_colosseum_final.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    json.dump({c["chunk_id"]: c["tags"] for c in final}, open(out / "tag_overrides.json", "w", encoding="utf-8"), indent=1)
    rep = [f"chunk finali: {len(final)} (scartati {len(chunks) - len(final)})", "", "copertura per tag:"]
    for t in TAGS:
        n = sum(t in c["tags"] for c in final); rep.append(f"  {t:13s} {n:3d}  {100 * n / len(final):4.0f}%")
    rep += ["", "chunk per documento:"] + [f"  {d:28s} {n}" for d, n in Counter(c["doc_id"] for c in final).items()]
    rep += ["", f"SENZA revisione manuale (tag provvisori): {pending}"]
    (out / "review_report.txt").write_text("\n".join(rep), encoding="utf-8"); print("\n".join(rep))

if __name__ == "__main__": main()
EOF

In [25]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/evaluation/make_queries.py
#!/usr/bin/env python3
"""make_queries.py - genera dal corpus Colosseo: queries_predetermined.json (~30%), queries_normal.json, eval_queries.json, gold_chunks.json.
Uso: python make_queries.py --chunks data/queries/colosseum/chunks_colosseum_final.json --out data/queries/colosseum [--seed 42]
Le 42 query sono definite qui sotto (testo, categoria, tag di conversazione, gold in ordine di importanza, risposta di riferimento)."""
import argparse, json, random
from pathlib import Path
P = {"wc": "wiki_colosseum", "da": "wiki_domus_aurea", "cn": "wiki_colossus_of_nero", "lm": "wiki_ludus_magnus", "vn": "wiki_venatio",
     "nm": "wiki_naumachia", "vl": "wiki_velarium", "ac": "wiki_arch_of_constantine", "pa": "paper_arena_floor", "ug": "web_underground"}
def g(*s): return [f"{P[x.split('_')[0]]}_{x.split('_')[1]}" for x in s]
H, A, S, R, C, U = "History", "Architecture", "Spectacles", "Archaeology", "Conservation", "Culture"

# id, categoria, sottotipo, query, expected_tag (tag di conversazione), tags (interessi del visitatore), gold, reference
Q = [
# ---------------- AMBIGUE: la stessa formulazione ha piu' letture; il tag di conversazione ne seleziona una
("AMB_01","ambiguous","modern_vs_ancient","What is the arena floor made of?",C,[],g("pa_10","pa_08","pa_07"),
 "The partially rebuilt arena floor uses laminated wood, reinforced with carbon-fibre and aramid-fibre where needed, assembled with stainless steel plates and bolts so that it can be dismounted (reversibility). The ancient floor was wood covered with sand."),
("AMB_02","ambiguous","build_vs_use","Who built the part under the arena, and when?",R,[],g("ug_03","ug_02","wc_23"),
 "The hypogeum was ordered by Domitian; by the dates stamped on the bricks the walls supporting the arena were first built about ten years after the inauguration (80 AD), and were later modified in four stages."),
("AMB_03","ambiguous","evidence_vs_accounts","How did water get into the arena?",R,[],g("ug_04","ug_03","wc_24"),
 "Before Domitian's hypogeum, the arena may have been filled with water; traces of 'locks' to make it waterproof up to about 5 feet and two spaces ('darsene') where ships were kept have been found. The accounts suggest a connection to a nearby aqueduct."),
("AMB_04","ambiguous","statue_vs_name","What happened to the Colossus?",H,[],g("cn_03","cn_02","da_09"),
 "The statue was moved by Hadrian next to the Colosseum, remained until at least the 4th-century sources, and then disappeared, possibly in the Sack of Rome in 410 or a 5th-century earthquake; only the pedestal foundations remain (pedestal removed in 1936, foundations excavated in 1986)."),
("AMB_05","ambiguous","name_meaning","Why is it called the Colosseum?",U,[],g("wc_04","cn_04"),
 "The name is believed to derive from the colossal statue of Nero (the Colossus) beside the amphitheatre; Bede's epigram about the Colossus is often mistranslated as referring to the Colosseum."),
("AMB_06","ambiguous","damage_cause","How was the building damaged over the centuries?",C,[],g("wc_12","wc_14","wc_17"),
 "Stone robbers stripped the interior and marble (burned for quicklime) and pried out the iron clamps, leaving pockmarks; vegetation, pollution and general deterioration threatened the structure, so the wall was shored up with brick wedges and restored."),
("AMB_07","ambiguous","paintings","What happened to the frescoes of the Golden House?",U,[],g("da_05","da_04","da_14"),
 "Rediscovered at the end of the 15th century, they profoundly influenced Renaissance artists (Raphael, Michelangelo studied them); the grotesque decorations came back in Neoclassicism, making the painter Famulus very influential."),
("AMB_08","ambiguous","living_quarters","Where did the gladiators live and train?",A,[],g("lm_08","lm_07","lm_09"),
 "At the Ludus Magnus: small rectangular chambers opening onto a two-storey portico served as living quarters, around an elliptical practice arena (about 63 x 42 m) with a small cavea."),
("AMB_09","ambiguous","site_today","Tell me about the Ludus Magnus.",R,[],g("lm_11","lm_06","lm_09"),
 "Rediscovered in 1937 and excavated in 1957-61, the remains represent less than half of the building and mostly reflect Trajan's rebuilding; the plan is completed with the Severan Marble Plan."),
("AMB_10","ambiguous","which_restoration","Tell me about the restoration.",C,[],g("wc_31","wc_14","pa_07"),
 "The Colosseum had a major 1993-2000 restoration and a 2013-2016 cleaning and repair financed by Tod's (EUR 25 million); a modern arena floor is being rebuilt to protect the hypogeum and give back the performance floor."),
("AMB_11","ambiguous","capacity","How many people could it hold?",A,[],g("wc_20","wc_01","wc_19"),
 "The Codex-Calendar of 354 gives 87,000, but modern estimates are around 50,000; other estimates range 50,000-80,000 with an average of about 65,000."),
("AMB_12","ambiguous","ships","Were there ships in the Colosseum?",S,[],g("wc_28","nm_09","nm_08"),
 "Ancient writers describe naumachiae (mock naval battles) at the inaugural games of 80 AD and under Domitian, but it is debated how the arena could be waterproofed and whether real ships could manoeuvre; stage-props representing ships may have been used."),
("AMB_13","ambiguous","sailors","What did the sailors do at the Colosseum?",A,[],g("vl_02","wc_18","vl_01"),
 "Sailors from the Misenum fleet, expert in sailmaking and rigging, built, maintained and operated the velarium, the retractable awning over the seats."),
("AMB_14","ambiguous","events_now","What kind of events take place at the Colosseum?",U,[],g("wc_15","wc_30","wc_02"),
 "Today only a few hundred spectators fit in temporary seating, so big concerts are held outside using it as a backdrop; religious ceremonies such as the Good Friday Way of the Cross take place there."),
# ---------------- SPECIALISTICHE: un solo dominio, domande su dettagli
("SPE_01","specialistic","dates","When did construction of the Colosseum begin and when were the inaugural games held?",H,[H],g("wc_09","wc_01"),
 "Construction began under Vespasian around 70-72 AD, the third storey was complete at his death in 79, Titus finished the top level in 80, and the inaugural games were held in 80 or 81 AD."),
("SPE_02","specialistic","crowd_flow","How many entrances did the Colosseum have and how did spectators find their seats?",A,[A],g("wc_19","wc_22"),
 "Eighty numbered entrances (76 for ordinary spectators); tickets were numbered pottery shards directing people to a section and row, and vomitoria let them reach seats and leave within minutes; each seat was designated by gradus, cuneus and number."),
("SPE_03","specialistic","animals","How many animals were killed at the inaugural games and which species appeared in the hunts?",S,[S],g("wc_10","vn_03","wc_27"),
 "Over 9,000 wild animals according to Dio Cassius; species included lions, elephants, bears, tigers, leopards, crocodiles, giraffes, hippos and ostriches."),
("SPE_04","specialistic","foundations","How deep are the foundations of the hypogeum walls and what were they built on?",R,[R],g("pa_05","pa_03"),
 "The foundations were dug between about 3.25 and 5.10 m below the hypogeal base floor, on a levelled concrete structure and a concrete platform 0.10-1.60 m thick, with the base rows of tufa blocks laid on top."),
("SPE_05","specialistic","sponsor","Who sponsored the 2011 restoration of the Colosseum and how much did it cost?",C,[C],g("wc_31"),
 "Diego Della Valle, head of Tod's, sponsored a EUR 25 million restoration; work began in 2013 and was completed on 1 July 2016."),
("SPE_06","specialistic","survey","What did the 1996 survey of the hypogea reveal about their history?",R,[R],g("ug_02","ug_05"),
 "Italian and German archaeologists surveyed the walls and divided the history of the hypogea in four stages: thin pillars, strengthening, further reinforcement, and finally division by the walls seen today, built with materials fallen from the top of the building."),
("SPE_07","specialistic","dimensions","How tall and wide is the Arch of Constantine?",A,[A],g("ac_01"),
 "21 m high, 25.9 m wide and 7.4 m deep; the central bay is 11.5 m high and 6.5 m wide."),
("SPE_08","specialistic","statue_move","When and how was the Colossus of Nero moved next to the Colosseum?",H,[H],g("cn_02","da_09"),
 "Around 128 AD Hadrian had it moved, by the architect Decrianus using 24 elephants, to make room for the Temple of Venus and Roma."),
("SPE_09","specialistic","augustus_naumachia","How many men and ships fought in Augustus's naumachia?",S,[S],g("nm_02"),
 "According to the Res Gestae, 3,000 men (not counting rowers) fought in 30 vessels with rams plus smaller boats, in a battle between Greeks and Persians staged in 2 BC."),
("SPE_10","specialistic","restoration_90s","Why was a major restoration carried out between 1993 and 2000 and how much did it cost?",C,[C],g("wc_14"),
 "The effects of pollution and general deterioration over time prompted it; it cost 40 billion lire (about EUR 20.6 million at 2000 prices)."),
("SPE_11","specialistic","lights","What does the colour of the Colosseum's night lighting mean?",U,[U],g("wc_15"),
 "The lighting changes from white to gold when someone condemned to death is released or a death sentence is commuted, or when a jurisdiction abolishes the death penalty (most recently Connecticut, November 2012)."),
("SPE_12","specialistic","velarium","How did the velarium work?",A,[A],g("wc_18","vl_02","vl_01"),
 "A retractable canvas awning, a net-like rope structure with a central hole, supported by 240 mast corbels at the top of the attic; it covered part of the arena, sloped towards the centre to catch the wind and create a breeze, and was operated by sailors from Misenum."),
("SPE_13","specialistic","sphinx_room","What was discovered in the Domus Aurea in 2018?",R,[R],g("da_18","da_19"),
 "A barrel-vaulted room (room 72), the Sphinx Room, decorated with panthers, centaurs, the god Pan and a sphinx in Fourth Style paintings, built between 65 and 68 AD."),
("SPE_14","specialistic","site_choice","Why did Vespasian build the amphitheatre on the site of Nero's lake?",H,[H],g("wc_08","wc_07","da_03"),
 "It was a populist gesture returning to the people an area Nero had appropriated; unlike most amphitheatres, built on city outskirts, it was placed at the heart of Rome."),
# ---------------- MULTI-DOMINIO: servono chunk di piu' domini/documenti
("MUL_01","multi_domain","damage_and_protection","How did earthquakes and stone robbing damage the Colosseum, and what has been done to protect it?",H,[H,C],g("wc_11","wc_12","wc_14","wc_31"),
 "The 1349 earthquake collapsed the south side and the stone was reused elsewhere; stripping removed marble and iron clamps. Later popes added brick wedges (1807, 1827), removed vegetation and repaired it; major restorations were done in 1993-2000 and 2013-2016."),
("MUL_02","multi_domain","flooding_evidence","Could the arena really be flooded for naval battles, and what evidence is there?",S,[S,R],g("wc_28","wc_24","ug_03","ug_04","nm_09"),
 "Ancient accounts describe flooding at the inaugural games, and traces of locks and 'docks' (darsene) were found, but there is no physical evidence of naumachiae and the arena was small; Domitian's hypogeum ended the practice."),
("MUL_03","multi_domain","social_seating","How were seats assigned to different social classes at the Colosseum?",A,[A,U],g("wc_20","wc_21","wc_22"),
 "Seating reflected Roman social hierarchy: emperor and Vestal Virgins in special boxes, senators on the podium, knights in the maenianum primum, citizens above (wealthy lower, poor upper), a top gallery for the poor, slaves and women; some groups were banned."),
("MUL_04","multi_domain","christian_view","How did the Christian view of the Colosseum change from the Middle Ages to the 18th century?",U,[H,U],g("wc_33","wc_34","wc_13","wc_32"),
 "In the Middle Ages it was treated as a quarry and not a sacred site; in the 17th-18th centuries, after Martinelli's book, popes declared it a sanctuary, Benedict XIV consecrated it and erected Stations of the Cross, though evidence of martyrdoms there is lacking."),
("MUL_05","multi_domain","floor_vs_hypogeum","How are the ancient hypogeum and the new arena floor reconciled in the reconstruction?",C,[R,C],g("pa_03","pa_07","pa_10","pa_08"),
 "The new wooden floor at the Flavian level rests on the original walls through interposed reversible elements, without occupying the hypogeal spaces, and protects the structures from weathering."),
("MUL_06","multi_domain","machinery","How did the underground machinery support the shows?",S,[S,A],g("wc_23","wc_24","ug_01","ug_05"),
 "Lifts, pulleys, trapdoors and hinged platforms raised animals and scenery from the hypogeum; the pegmata were prepared in the Summum Choragium and emerged in the arena to create instant scenery."),
("MUL_07","multi_domain","nero_to_flavian","How are the Domus Aurea, the Colossus and the Colosseum connected?",H,[H,A],g("wc_07","da_03","da_09","cn_02"),
 "After the fire of 64 Nero built the Domus Aurea with an artificial lake and the Colossus; Vespasian filled the lake and built the amphitheatre there, and Hadrian moved the Colossus next to it, giving the Colosseum its name."),
("MUL_08","multi_domain","games_evolution","How did the games at the Colosseum evolve from the inauguration to the last mentions?",S,[S,H],g("wc_10","wc_27","vn_02"),
 "From the 80 AD inaugural games with over 9,000 animals killed, through gladiators, hunts and executions, to Honorius's bans in 399 and 404; gladiatorial fights are last mentioned around 435 and animal hunts continued until at least 523."),
("MUL_09","multi_domain","vulnerability","What features of the Colosseum's structure and materials make it vulnerable to damage?",A,[A,C],g("wc_16","wc_12","wc_11","wc_17"),
 "The outer wall was travertine set without mortar and held by iron clamps (300 tons), so stripping the clamps and earthquakes caused large collapses, especially on the south side on alluvial terrain."),
("MUL_10","multi_domain","gladiator_route","Where did gladiators train and how did they reach the arena?",S,[S,R],g("lm_02","lm_03","wc_23","wc_25"),
 "They trained at the Ludus Magnus east of the Colosseum and reached the arena through an underground passage connecting it to the hypogeum."),
("MUL_11","multi_domain","plants_and_use","How do plants and modern events affect the preservation of the Colosseum?",C,[C,U],g("wc_36","wc_14","wc_15"),
 "Vegetation (684 species recorded) threatened the masonry and was removed, though much returned; because of the ruined interior only a few hundred spectators can be seated, so large concerts are held outside."),
("MUL_12","multi_domain","lost_and_found","What do we know about how the Domus Aurea was lost and then found again?",H,[H,R],g("da_02","da_03","da_04","da_06"),
 "Stripped of marble within a decade of Nero's death and built over by the Flavians, it survived underground; found in the late 15th century, it then suffered moisture, collapses and closures."),
("MUL_13","multi_domain","arch_spolia","How did the Arch of Constantine reuse earlier monuments and why is it controversial?",A,[A,U],g("ac_02","ac_06","ac_09","ac_14"),
 "Many reliefs and statues come from monuments of Trajan, Hadrian and Marcus Aurelius with portrait heads replaced; scholars debate whether it was built for Constantine, Maxentius or earlier, and art historians discuss the mixed styles."),
("MUL_14","multi_domain","octagonal_room","Why is the Octagonal Room important for archaeoastronomy and later architecture?",A,[R,A],g("da_15","da_17","da_16"),
 "Its domed octagonal design with a central oculus may have been a model for the Pantheon, and its openings are calibrated so that on the equinoxes the beam of light strikes the northern doorway."),
]

def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--chunks", required=True); ap.add_argument("--out", required=True)
    ap.add_argument("--seed", type=int, default=42); ap.add_argument("--per_cat", type=int, default=4)
    a = ap.parse_args()
    ids = {c["chunk_id"] for c in json.load(open(Path(a.chunks).expanduser(), encoding="utf-8"))}
    out = Path(a.out).expanduser(); out.mkdir(parents=True, exist_ok=True)
    qs, gold = [], {}
    for qid, cat, sub, text, exp, tags, gd, ref in Q:
        missing = [c for c in gd if c not in ids]
        assert not missing, f"{qid}: gold inesistenti {missing}"
        assert len(gd) == len(set(gd)), f"{qid}: gold duplicati"
        qs.append({"id": qid, "category": cat, "subtype": sub, "query": text, "expected_tag": exp, "tags": tags, "reference": ref})
        gold[qid] = gd
    assert len({q["id"] for q in qs}) == len(qs)
    rng = random.Random(a.seed); pred_ids = set()
    for cat in ("ambiguous", "specialistic", "multi_domain"):
        pool = [q["id"] for q in qs if q["category"] == cat]
        pred_ids |= set(rng.sample(pool, a.per_cat))
    pred = [q for q in qs if q["id"] in pred_ids]; norm = [q for q in qs if q["id"] not in pred_ids]
    dump = lambda name, obj: json.dump(obj, open(out / name, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    dump("queries_predetermined.json", pred); dump("queries_normal.json", norm)
    dump("eval_queries.json", [{**q, "is_predetermined": q["id"] in pred_ids} for q in qs]); dump("gold_chunks.json", gold)
    print(f"query: {len(qs)} (predeterminate {len(pred)} = {100*len(pred)/len(qs):.0f}%, normali {len(norm)}), seed={a.seed}")
    print("predeterminate:", sorted(pred_ids))
    from collections import Counter
    print("chunk gold usati:", len({c for v in gold.values() for c in v}), "di", len(ids))
    print("tag di conversazione:", dict(Counter(q["expected_tag"] for q in qs)))

if __name__ == "__main__": main()
EOF

In [ ]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/evaluation/colosseum_curation.py
"""
Taratura "da esperto di dominio" del grafo Colosseo (scritta a mano leggendo i chunk, non derivata dal gold).
Salvare come ~/tesi_graphrag/src/evaluation/colosseum_curation.py   (stessa struttura di armstrong_curation.py)

GROUPS: gruppi di chunk che un gestore del sito collega perche' raccontano lo stesso episodio/tema, INDIPENDENTEMENTE dalle query.
        "core" = nucleo (coppie in classe FORTE); gli altri "members" si collegano a tutti in classe LIEVE.
CROSS : gruppi trasversali ai documenti (stesso tipo di informazione: restauri, dimensioni, scavi...). Disattivabili.
PREDQ : per ogni predeterminata, interventi del gestore guardando la query:
        support = chunk non gold ma utili (avvicinati lievi); away = chunk di una lettura DIVERSA dell'ambiguita' (s = forte, m = lieve).
Prefissi: wc Colosseum, da Domus Aurea, cn Colossus of Nero, lm Ludus Magnus, vn Venatio, nm Naumachia, vl Velarium,
          ac Arch of Constantine, pa articolo Cerone (pavimento dell'arena), ug sotterranei (the-colosseum.net)
"""
_P = {"wc": "wiki_colosseum", "da": "wiki_domus_aurea", "cn": "wiki_colossus_of_nero", "lm": "wiki_ludus_magnus", "vn": "wiki_venatio",
      "nm": "wiki_naumachia", "vl": "wiki_velarium", "ac": "wiki_arch_of_constantine", "pa": "paper_arena_floor", "ug": "web_underground"}
def K(*s): return [f"{_P[x.split('_')[0]]}_{x.split('_')[1]}" for x in s]

GROUPS = {
 # --- storia del Colosseo
 "name_origin":        (K("wc_03","wc_04"),                      K("cn_04","da_09","wc_02")),
 "site_and_flavians":  (K("wc_07","wc_08","da_03"),              K("da_02","wc_09","cn_01")),
 "building_inaug":     (K("wc_09","wc_10"),                      K("wc_01","wc_08")),
 "late_antiquity":     (K("wc_10","wc_11"),                      K("wc_12","ug_05","pa_03")),
 "medieval_reuse":     (K("wc_11","wc_12"),                      K("wc_13","wc_33","wc_02")),
 "christian_site":     (K("wc_32","wc_33","wc_34","wc_35"),      K("wc_13","wc_02")),
 "modern_restoration": (K("wc_14","wc_31"),                      K("wc_26","wc_30","wc_12","pa_07")),
 "plants_and_decay":   (K("wc_36","wc_14"),                      K("wc_17")),
 "modern_use":         (K("wc_15","wc_30","wc_35"),              K("wc_02","wc_31","wc_37")),
 # --- architettura
 "exterior":           (K("wc_16","wc_17","wc_18"),              K("wc_19","vl_01","vl_02")),
 "velarium":           (K("wc_18","vl_01","vl_02"),              K("wc_16")),
 "crowd_and_seating":  (K("wc_19","wc_20","wc_21","wc_22"),      K("wc_01")),
 "arena_hypogeum":     (K("wc_23","wc_24","ug_02","ug_05"),      K("ug_01","ug_03","ug_04","pa_01","pa_05")),
 # --- spettacoli
 "games_and_hunts":    (K("wc_27","vn_02","vn_03"),              K("vn_04","vn_05","wc_29","wc_10")),
 "executions":         (K("wc_29","vn_05","wc_27"),              K("vn_04")),
 "arena_flooding":     (K("wc_28","wc_24","ug_03","ug_04","nm_09"), K("nm_08","nm_10","nm_11")),
 "naumachia_roman":    (K("nm_02","nm_03","nm_05","nm_06"),      K("nm_01","nm_07","nm_08")),
 "naumachia_amph":     (K("nm_08","nm_09","nm_10","nm_11"),      K("nm_12","nm_13","nm_14","wc_28")),
 "naumachia_decline":  (K("nm_12","nm_13","nm_14"),              K("nm_09")),
 "gladiator_school":   (K("lm_01","lm_03","lm_02","wc_25"),      K("lm_04","lm_08","lm_10","wc_23")),
 "ludus_remains":      (K("lm_06","lm_11","lm_07","lm_08","lm_09"), K("lm_02","lm_04")),
 # --- Nerone e Domus Aurea
 "colossus":           (K("cn_01","cn_02","cn_03"),              K("da_09","da_08","wc_04","cn_04")),
 "domus_construction": (K("da_02","da_07","da_08","da_11"),      K("da_01","da_10","da_13")),
 "domus_decoration":   (K("da_13","da_14","da_19","da_20"),      K("da_05","da_18")),
 "domus_rediscovery":  (K("da_04","da_05"),                      K("da_06","da_14")),
 "domus_decay":        (K("da_06","da_20"),                      K("da_04")),
 "octagonal_room":     (K("da_15","da_16","da_17"),              K("da_11")),
 "sphinx_room":        (K("da_18","da_19","da_20"),              []),
 # --- Arco di Costantino
 "arch_commission":    (K("ac_01","ac_03","ac_04","ac_06","ac_07"), K("ac_05","ac_08")),
 "arch_sculpture":     (K("ac_02","ac_09","ac_10","ac_13","ac_14"), K("ac_16","ac_18","ac_19","ac_20")),
 "arch_decoration":    (K("ac_16","ac_18","ac_19","ac_20","ac_25"), K("ac_14")),
 # --- pavimento dell'arena (Cerone et al.)
 "arena_reconstruction": (K("pa_01","pa_02","pa_07","pa_08","pa_10"), K("pa_03","pa_06","wc_31")),
 "arena_foundations":    (K("pa_03","pa_04","pa_05","pa_06"),         K("ug_02","ug_03","wc_23")),
}

CROSS = {
 "X_restorations":  (K("wc_31","da_06","ac_05"),         K("wc_14","pa_07")),
 "X_dimensions":    (K("wc_16","ac_01","lm_07"),         K("cn_01","wc_23")),
 "X_excavations":   (K("da_04","lm_06","cn_03"),         K("pa_01","wc_14")),
 "X_sun_statue":    (K("cn_02","da_09"),                 K("ac_08","wc_04")),
 "X_mussolini":     (K("wc_14","cn_03"),                 K("lm_06")),
 "X_ancient_texts": (K("da_07","nm_08"),                 K("lm_10","da_14","wc_10")),
 "X_hadrian":       (K("cn_02","da_09"),                 K("ac_19","lm_04")),
}

PREDQ = {
 "AMB_01": {"support": K("pa_02"),               "away": {**{c: "s" for c in K("wc_23")}, **{c: "m" for c in K("ug_03")}}},
 "AMB_02": {"support": K("wc_10"),               "away": {c: "m" for c in K("wc_30")}},
 "AMB_03": {"support": K("nm_09"),               "away": {c: "m" for c in K("wc_28","nm_10")}},
 "AMB_04": {"support": K("cn_01"),               "away": {**{c: "s" for c in K("wc_04")}, **{c: "m" for c in K("cn_04")}}},
 "AMB_05": {"support": K("wc_03"),               "away": {c: "m" for c in K("cn_03","cn_02")}},
 "AMB_06": {"support": K("wc_11","wc_10"),       "away": {}},
 "AMB_07": {"support": K("da_13"),               "away": {**{c: "s" for c in K("da_06")}, **{c: "m" for c in K("da_20")}}},
 "AMB_08": {"support": K("lm_02","lm_01"),       "away": {c: "m" for c in K("lm_11","lm_06","wc_25")}},
 "AMB_09": {"support": K("lm_07","lm_08"),       "away": {c: "m" for c in K("lm_01","lm_03")}},
 "AMB_10": {"support": K("wc_26","ac_05"),       "away": {}},
 "AMB_11": {"support": K("wc_21"),               "away": {c: "m" for c in K("lm_03")}},
 "AMB_12": {"support": K("nm_10","nm_11","ug_04"), "away": {c: "m" for c in K("vl_02","wc_18")}},
 "AMB_13": {"support": [],                       "away": {**{c: "s" for c in K("wc_28")}, **{c: "m" for c in K("nm_09")}}},
 "AMB_14": {"support": K("wc_35","wc_31"),       "away": {**{c: "s" for c in K("wc_27")}, **{c: "m" for c in K("wc_29")}}},
 "SPE_01": {"support": K("wc_10"),               "away": {}},
 "SPE_02": {"support": K("wc_21","wc_20"),       "away": {}},
 "SPE_03": {"support": K("vn_02","vn_04"),       "away": {}},
 "SPE_04": {"support": K("pa_04","ug_02"),       "away": {}},
 "SPE_05": {"support": K("wc_14"),               "away": {}},
 "SPE_06": {"support": K("ug_03"),               "away": {}},
 "SPE_07": {"support": K("ac_03"),               "away": {}},
 "SPE_08": {"support": K("cn_03","cn_01"),       "away": {}},
 "SPE_09": {"support": K("nm_05","nm_06"),       "away": {}},
 "SPE_10": {"support": K("wc_31"),               "away": {}},
 "SPE_11": {"support": K("wc_14"),               "away": {}},
 "SPE_12": {"support": [],                       "away": {}},
 "SPE_13": {"support": K("da_20"),               "away": {}},
 "SPE_14": {"support": K("da_02"),               "away": {}},
 "MUL_01": {"support": K("wc_10","wc_26"),       "away": {}},
 "MUL_02": {"support": K("nm_10","nm_08","wc_23"), "away": {}},
 "MUL_03": {"support": K("wc_19"),               "away": {}},
 "MUL_04": {"support": K("wc_35","wc_02"),       "away": {}},
 "MUL_05": {"support": K("pa_02","pa_01","pa_06"), "away": {}},
 "MUL_06": {"support": K("wc_30","ug_02"),       "away": {}},
 "MUL_07": {"support": K("cn_01","da_02","da_08"), "away": {}},
 "MUL_08": {"support": K("vn_05","wc_29","wc_28"), "away": {}},
 "MUL_09": {"support": K("wc_14"),               "away": {}},
 "MUL_10": {"support": K("lm_01","wc_24"),       "away": {}},
 "MUL_11": {"support": K("wc_30","wc_31"),       "away": {}},
 "MUL_12": {"support": K("da_01","da_07"),       "away": {}},
 "MUL_13": {"support": K("ac_07","ac_08","ac_10"), "away": {}},
 "MUL_14": {"support": K("da_11"),               "away": {}},
}
EOF

In [ ]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/evaluation/add_monument_tags.py
#!/usr/bin/env python3
"""add_monument_tags.py - struttura a DUE ASSI per il benchmark Colosseo.
Salvare come ~/tesi_graphrag/src/evaluation/add_monument_tags.py

  - ogni CHUNK riceve un tag MONUMENTO (derivato da doc_id, mappa MONUMENT sotto) in testa alla lista, piu' i tag tematici di prima;
  - ogni QUERY ha SEMPRE un conversation tag = il MONUMENTO della conversazione (expected_tag); l'ASPETTO (ex expected_tag
    tematico) passa nei tag manuali (`tags`), insieme a quelli che c'erano gia'.
  - regola per le query il cui gold attraversa piu' monumenti: monumento = quello del PRIMO chunk gold (il piu' importante);
    gli altri monumenti restano raggiungibili solo con tag manuali / distanze (e' proprio dove le distanze servono).
  - PIU' conv tag: le query AMBIGUE (aspetto unico, stabilito dalla conversazione) hanno conv_tag = [monumento, aspetto]; la categoria 1
    richiede TUTTI i conv tag, quindi il pool e' ristretto a (monumento AND aspetto). Eccezioni in MULTI_CONV_EXCLUDE: query il cui gold
    attraversa monumenti/aspetti (AMB_04, AMB_05): conv_tag = [monumento], aspetto nei tag manuali. Specialistiche e multi-dominio:
    conv_tag = [monumento], aspetti nei tag manuali (le multi-dominio hanno piu' aspetti: un conv_tag in piu' escluderebbe gold dal pool).
Predeterminate e normali: stessa estrazione di make_queries.py (stesso seed), quindi gli stessi id del piano di distanze.

Uso: python add_monument_tags.py --chunks data/queries/colosseum/chunks_colosseum_final.json --out data/queries/colosseum_v2 [--seed 42]
"""
import argparse, json, random
from collections import Counter
from pathlib import Path

# NOMI SENZA SPAZI (stessa convenzione dei tag precedenti): cambiali qui se vuoi etichette diverse nell'interfaccia.
MONUMENT = {
    "wiki_colosseum": "Colosseum", "paper_arena_floor": "Colosseum", "web_underground": "Colosseum",
    "wiki_velarium": "Colosseum", "wiki_venatio": "Colosseum", "wiki_naumachia": "Colosseum",
    "wiki_domus_aurea": "DomusAurea", "wiki_colossus_of_nero": "ColossusOfNero",
    "wiki_ludus_magnus": "LudusMagnus", "wiki_arch_of_constantine": "ArchOfConstantine",
}
MONUMENTS = sorted(set(MONUMENT.values()))
MULTI_CONV_CATEGORIES = ("ambiguous",)       # categorie con conv_tag = [monumento, aspetto]
MULTI_CONV_EXCLUDE = {"AMB_04", "AMB_05"}    # ambigue il cui gold attraversa monumenti/aspetti: un solo conv_tag (modificabile)


def tag_chunks(chunks):
    out = []
    for c in chunks:
        m = MONUMENT[c["doc_id"]]                       # KeyError = doc_id nuovo da mappare: voluto
        out.append({**c, "tags": [m] + [t for t in c["tags"] if t != m]})
    return out


def restructure_queries(queries, gold, chunk_monument, multi_cats=MULTI_CONV_CATEGORIES, exclude=MULTI_CONV_EXCLUDE):
    """expected_tag := [monumento del primo gold] (+ [aspetto] per le categorie multi_cats, tranne exclude);
    tags := aspetto precedente + manuali precedenti (senza duplicati), tolti quelli gia' passati nei conv tag."""
    out = []
    for q in queries:
        g = gold[q["id"]]
        mon = chunk_monument[g[0]]
        aspects = list(dict.fromkeys(([q["expected_tag"]] if q.get("expected_tag") else []) + list(q.get("tags", []))))
        spread = sorted({chunk_monument[x] for x in g})
        conv = [mon]
        if q["category"] in multi_cats and q["id"] not in exclude and q.get("expected_tag"):
            conv.append(q["expected_tag"]); aspects = [x for x in aspects if x not in conv]
        out.append({**q, "expected_tag": conv, "tags": aspects, "aspect_tag": q.get("expected_tag"),
                    "gold_monuments": spread})
    return out


def main():
    import importlib.util, sys
    ap = argparse.ArgumentParser(); ap.add_argument("--chunks", required=True); ap.add_argument("--out", required=True)
    ap.add_argument("--seed", type=int, default=42); ap.add_argument("--per_cat", type=int, default=4)
    ap.add_argument("--make_queries", default=str(Path(__file__).with_name("make_queries.py")))
    a = ap.parse_args()
    spec = importlib.util.spec_from_file_location("make_queries", a.make_queries)
    mq = importlib.util.module_from_spec(spec); spec.loader.exec_module(mq)

    chunks = json.load(open(Path(a.chunks).expanduser(), encoding="utf-8"))
    chunks2 = tag_chunks(chunks)
    cm = {c["chunk_id"]: c["tags"][0] for c in chunks2}
    out = Path(a.out).expanduser(); out.mkdir(parents=True, exist_ok=True)

    qs, gold = [], {}
    for qid, cat, sub, text, exp, tags, gd, ref in mq.Q:
        assert all(x in cm for x in gd), f"{qid}: gold inesistenti"
        qs.append({"id": qid, "category": cat, "subtype": sub, "query": text, "expected_tag": exp, "tags": tags, "reference": ref})
        gold[qid] = gd
    qs = restructure_queries(qs, gold, cm)

    rng = random.Random(a.seed); pred_ids = set()                       # stessa estrazione di make_queries.main()
    for cat in ("ambiguous", "specialistic", "multi_domain"):
        pred_ids |= set(rng.sample([q["id"] for q in qs if q["category"] == cat], a.per_cat))
    pred = [q for q in qs if q["id"] in pred_ids]; norm = [q for q in qs if q["id"] not in pred_ids]
    dump = lambda name, obj: json.dump(obj, open(out / name, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    dump("chunks_colosseum_final.json", chunks2)
    dump("queries_predetermined.json", pred); dump("queries_normal.json", norm)
    dump("eval_queries.json", [{**q, "is_predetermined": q["id"] in pred_ids} for q in qs]); dump("gold_chunks.json", gold)

    print("predeterminate:", sorted(pred_ids))
    print("chunk per monumento:", dict(Counter(cm.values())))
    print("monumento (primo conv_tag):", dict(Counter(q["expected_tag"][0] for q in qs)))
    print("query con piu' conv tag:", sum(len(q["expected_tag"]) > 1 for q in qs), "di", len(qs))
    print("\nid       conv_tag                            tag manuali                  pool cat.1  gold su altri monumenti")
    for q in qs:
        other = [m for m in q["gold_monuments"] if m != q["expected_tag"][0]]
        pool = sum(set(q["expected_tag"]) <= set(c["tags"]) for c in chunks2)
        print(f"{q['id']:7s}  {'+'.join(q['expected_tag']):34s}  {','.join(q['tags']):26s}  {pool:4d}      {','.join(other) or '-'}")


if __name__ == "__main__":
    main()
EOF

In [32]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/evaluation/colosseum_controls.py
"""
colosseum_controls.py - controlli SOLO RETRIEVAL (nessun LLM, nessun giudice) per il benchmark Colosseo.
Salvare come ~/tesi_graphrag/src/evaluation/colosseum_controls.py

Quattro controlli, tutti riprendibili (stesso run_id = riprende da dove era arrivato, come bench.generate):
  1. wrong_tag      - controllo NEGATIVO: stesse query con un conv_tag SBAGLIATO. Se il tag agisce in modo causale il recall deve calare.
  2. two_readings   - la stessa query con due letture (tag A / tag B, gold A / gold B): il sistema segue il tag o lo ignora?
  3. degradation    - degradazione dei tag separata (mancanti / sbagliati / solo primo tag / combinata) su piu' seed.
  4. distance       - le distanze servono davvero? sidecar CON distanze contro sidecar SENZA distanze (stessi tag), al variare di top_k.

PERCHE' IL CONTROLLO 4 (letto da distance_tag_reranker.py / distance_retriever.py):
  - il 2° retrieval (distanze) parte SOLO se esistono chunk di categoria 1 (chunk del 1° retrieval con il conv_tag) e,
    senza flag JOIN_*, riempie solo i posti rimasti: top_m = top_j - n_categoria1. Con top_k=15 e un tag frequente la
    categoria 1 riempie gia' i top_j e il 2° retrieval non parte (nei tuoi dati C1 e C3 danno lo stesso insieme in 26 query normali su 30).
  - le categorie 1-4 sono definite dal conv_tag a prescindere da order_by: C2_dist NON e' "solo distanze", e' "conv_tag + ordine per DIST".
  - con JOIN_* il distance_factor delle categorie 2-3 viene azzerato (=1.0): la classe di distanza decide solo CHI ENTRA nel pool.
    Quindi C5 senza DIST sarebbe identico a C5: l'ablazione pulita e' "sidecar senza distanze" (controllo 4).
"""
import json
import random
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd

BASE_KEYS = ["query_id", "config"]

# ------------------------------------------------------------------------------------------------ utilita'
def _boot(diffs, n_boot=5000, seed=0):
    diffs = np.asarray(diffs, dtype=float)
    if len(diffs) == 0:
        return float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    means = rng.choice(diffs, size=(n_boot, len(diffs)), replace=True).mean(axis=1)
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


@contextmanager
def _patched(bench, queries=None, configs=None, top_k=None, top_j=None):
    """Sostituisce temporaneamente query / configurazioni / top_k / top_j del benchmark e li ripristina sempre."""
    old = (bench.queries, bench.configs, bench.top_k, bench.top_j)
    try:
        if queries is not None:
            bench.queries = queries
        if configs is not None:
            bench.configs = [c for c in old[1] if c["name"] in configs]
        if top_k is not None:
            bench.top_k = top_k
        if top_j is not None:
            bench.top_j = top_j
        yield bench
    finally:
        bench.queries, bench.configs, bench.top_k, bench.top_j = old


def run(bench, run_id, queries=None, configs=None, top_k=None, top_j=None):
    """Retrieval di tutte le query x config (senza LLM). Restituisce il DataFrame di generazione del run."""
    with _patched(bench, queries, configs, top_k, top_j):
        return bench.generate(run_id, with_answers=False)


def run_regime(bench, run_id, top_k, top_j, queries=None, configs=None, with_answers=False):
    """Un REGIME (top_k, top_j) con confronto alla pari: le config di reranking usano (top_k, top_j), mentre C0 usa top_k = top_j
    (vettoriale puro a top_j chunk). Senza questo, con top_k < top_j C0 vedrebbe solo top_k chunk invece di top_j.
    Salva due file (<run_id>_rr, <run_id>_c0) e restituisce il DataFrame unito."""
    names = [c["name"] for c in bench.configs if configs is None or c["name"] in configs]
    rr = [n for n in names if n != "C0_baseline"]
    parts = []
    with _patched(bench, queries, rr, top_k, top_j):
        parts.append(bench.generate(f"{run_id}_rr", with_answers=with_answers))
    if "C0_baseline" in names:
        with _patched(bench, queries, ["C0_baseline"], max(top_k, top_j), top_j):
            parts.append(bench.generate(f"{run_id}_c0", with_answers=with_answers))
    return pd.concat(parts, ignore_index=True)


def short_context_quota(gen, top_j):
    """Quota di query (per config) con meno di top_j chunk nel contesto: categoria 1 vuota, il 2° retrieval non e' partito
    e il contesto resta quello dei soli top_k vettoriali. Con top_k < top_j e' il caso da contare."""
    n = gen.retrieved_ids.apply(len)
    return (n < top_j).groupby(gen.config).mean().round(2)


def gold_metrics(ids, gold_ids):
    g = set(gold_ids)
    hits = [i for i, c in enumerate(ids) if c in g]
    return {"recall": len(g & set(ids)) / len(g), "mrr": 1.0 / (hits[0] + 1) if hits else 0.0}


def _mask(df, name):
    normal = ~df.is_predetermined.astype(bool)
    return {"normali": normal,
            "ambigue normali": normal & (df.category == "ambiguous"),
            "tutte": pd.Series(True, index=df.index)}[name]


SUBSETS = ("normali", "ambigue normali", "tutte")


def paired_between_runs(gen_a, gen_b, metrics=("gold_recall", "gold_mrr"), subsets=SUBSETS, tol=0.01):
    """Differenza appaiata (B - A) sulle STESSE query e config, con IC95% bootstrap e win/tie/loss."""
    m = gen_a.merge(gen_b, on=BASE_KEYS, suffixes=("_a", "_b"))
    m["is_predetermined"], m["category"] = m["is_predetermined_a"], m["category_a"]
    rows = []
    for s in subsets:
        sub = m[_mask(m, s)]
        for cfg, g in sub.groupby("config"):
            for met in metrics:
                d = (g[f"{met}_b"] - g[f"{met}_a"]).to_numpy(float)
                lo, hi = _boot(d)
                rows.append({"subset": s, "config": cfg, "metric": met, "n": len(d),
                             "mean_A": round(g[f"{met}_a"].mean(), 3), "mean_B": round(g[f"{met}_b"].mean(), 3),
                             "diff_B_minus_A": round(d.mean(), 3), "ci95_low": round(lo, 3), "ci95_high": round(hi, 3),
                             "win": int((d > tol).sum()), "tie": int((abs(d) <= tol).sum()), "loss": int((d < -tol).sum())})
    return pd.DataFrame(rows)


# ------------------------------------------------------------------------------------------------ 1. tag sbagliato
MONUMENT_TAGS = ("Colosseum", "DomusAurea", "ColossusOfNero", "LudusMagnus", "ArchOfConstantine")   # come in add_monument_tags.py


def conv_of(q):
    """Lista dei conv tag della query (expected_tag puo' essere stringa o lista): il primo e' il monumento, gli altri aspetti."""
    e = q.get("expected_tag")
    return [e] if isinstance(e, str) else list(e or [])


def wrong_tag_queries(queries, gold, chunks, tags, seed=0, mode="monument"):
    """Copia delle query con un tag SBAGLIATO su uno dei due assi (si sceglie quello meno presente nei chunk gold; a parita' random col seed):
      mode="monument": il monumento (primo conv_tag) e' un ALTRO monumento; gli aspetti restano giusti;
      mode="aspect"  : il monumento resta giusto; l'aspetto (conv_tag aggiuntivo o, se non c'e', tag manuale) diventa un aspetto sbagliato.
    Confronto corretto: sulle query NORMALI (le predq hanno comunque le adiacenze query->chunk, che non dipendono dal tag)."""
    rng = random.Random(seed)
    mons = [t for t in tags if t in MONUMENT_TAGS]; asp = [t for t in tags if t not in MONUMENT_TAGS]
    if not mons:
        raise ValueError("TAGS non contiene tag monumento: stai usando i chunk v1 (09C_06)? Esegui prima le celle 0-4 di 09C_08 "
                         "nello STESSO kernel (chunks/gold/TAGS/bench devono essere quelli v2), poi riesegui la cella 0 di 09C_07.")
    out = []
    for q in queries:
        conv = conv_of(q)
        gt = [x for cid in gold[q["id"]] for x in chunks[cid]["tags"]]
        if mode == "monument":
            cands = sorted(set(mons) - {conv[0]}, key=lambda x: (gt.count(x), rng.random()))
            out.append({**q, "orig_expected_tag": conv, "expected_tag": [cands[0]] + conv[1:]})
        elif len(conv) > 1:
            cands = sorted(set(asp) - set(conv), key=lambda x: (gt.count(x), rng.random()))
            out.append({**q, "orig_expected_tag": conv, "expected_tag": [conv[0], cands[0]]})
        else:
            cands = sorted(set(asp) - set(q["tags"]), key=lambda x: (gt.count(x), rng.random()))
            out.append({**q, "orig_tags": list(q["tags"]), "tags": [cands[0]] if q["tags"] else []})
    return out


def wrong_tag_leak(gen, queries, chunks, mode="monument"):
    """Quota dei top-J che porta il tag (sbagliato) usato: se alta, il sistema SEGUE il tag."""
    use = {q["id"]: (conv_of(q)[:1] if mode == "monument" else conv_of(q)[1:] + list(q["tags"])) for q in queries}
    g = gen.copy()
    g["quota_con_tag_usato"] = [np.mean([bool(set(use[r.query_id]) & set(chunks[i]["tags"])) for i in r.retrieved_ids])
                                for r in g.itertuples()]
    return g.groupby("config").quota_con_tag_usato.mean().round(2)


# ------------------------------------------------------------------------------------------------ 2. due letture
# BOZZA da rivedere leggendo i chunk: per ogni ambigua una seconda lettura (tag B, gold B). Il gold A e' quello di gold_chunks.json.
TWO_READINGS = {
    "AMB_14": {"tag_b": "Spectacles",   "gold_b": ["wiki_colosseum_27", "wiki_colosseum_10", "wiki_colosseum_29"]},      # oggi (Culture) vs giochi antichi
    "AMB_12": {"tag_b": "Archaeology",  "gold_b": ["web_underground_04", "web_underground_03", "wiki_colosseum_24"]},    # resoconti (Spectacles) vs tracce fisiche
    "AMB_07": {"tag_b": "Conservation", "gold_b": ["wiki_domus_aurea_06", "wiki_domus_aurea_20"]},                      # influenza sul Rinascimento vs degrado
    "AMB_09": {"tag_b": "History",      "gold_b": ["wiki_ludus_magnus_04", "wiki_ludus_magnus_05", "wiki_ludus_magnus_01"]},  # scavi (Archaeology) vs storia
    "AMB_08": {"tag_b": "Spectacles",   "gold_b": ["wiki_ludus_magnus_03", "wiki_ludus_magnus_10", "wiki_ludus_magnus_01"]},  # edificio (Architecture) vs vita/addestramento
}


def validate_two_readings(pairs, gold, chunks):
    """Controlli di coerenza: chunk esistenti, tag B presente nei gold B, nessuna sovrapposizione gold A / gold B."""
    for qid, p in pairs.items():
        assert all(c in chunks for c in p["gold_b"]), f"{qid}: gold_b con chunk inesistenti"
        assert not set(gold[qid]) & set(p["gold_b"]), f"{qid}: gold A e gold B si sovrappongono"
        miss = [c for c in p["gold_b"] if p["tag_b"] not in chunks[c]["tags"]]
        assert not miss, f"{qid}: tag_b '{p['tag_b']}' assente in {miss}"
    return True


def two_readings_queries(queries, pairs):
    by_id = {q["id"]: q for q in queries}
    out = []
    for qid, p in pairs.items():
        q = by_id[qid]
        out.append({**q, "id": f"{qid}__A", "is_predetermined": False})
        conv = conv_of(q)
        if len(conv) > 1:   # l'aspetto e' un conv tag: la lettura B lo sostituisce (stesso monumento)
            out.append({**q, "id": f"{qid}__B", "expected_tag": [conv[0], p["tag_b"]], "is_predetermined": False})
        else:               # l'aspetto e' un tag manuale
            out.append({**q, "id": f"{qid}__B", "tags": [p["tag_b"]], "is_predetermined": False})
    return out


def two_readings_report(gen, pairs, gold):
    """Per config: recall del gold della lettura VOLUTA (own), recall del gold dell'altra lettura (other), specificita'.
    specificita' = media di [(R_own - R_other)] sulle due direzioni. 0 = il tag non decide la lettura (C0 vale 0 per costruzione)."""
    rows = []
    for cfg, g in gen.groupby("config"):
        ids = {r.query_id: list(r.retrieved_ids) for r in g.itertuples()}
        for qid, p in pairs.items():
            ga, gb = gold[qid], p["gold_b"]
            for side, own, other in (("A", ga, gb), ("B", gb, ga)):
                r = ids[f"{qid}__{side}"]
                rows.append({"config": cfg, "query": qid, "lettura": side,
                             "R_own": gold_metrics(r, own)["recall"], "R_other": gold_metrics(r, other)["recall"]})
    d = pd.DataFrame(rows)
    d["specificita"] = d.R_own - d.R_other
    return d.groupby("config")[["R_own", "R_other", "specificita"]].mean().round(3), d


# ------------------------------------------------------------------------------------------------ 3. degradazione tag
# (parametri ASPETTO, parametri MONUMENTO). Valori arbitrari ma dichiarati: il monumento e' facile da assegnare (si legge dal documento),
# quindi nel caso "utente medio" ha errori piu' bassi dell'aspetto.
VARIANTS = {
    "tag_esperti":           (dict(), dict()),
    "aspetti_mancanti30":    (dict(missing_frac=0.30), dict()),
    "aspetti_sbagliati15":   (dict(wrong_frac=0.15), dict()),
    "aspetti_sbagliati30":   (dict(wrong_frac=0.30), dict()),
    "solo_primo_aspetto":    (dict(drop_secondary=True), dict()),
    "monumento_mancante20":  (dict(), dict(missing_frac=0.20)),
    "monumento_sbagliato10": (dict(), dict(wrong_frac=0.10)),
    "combinato_user":        (dict(missing_frac=0.30, wrong_frac=0.15, drop_secondary=True), dict(missing_frac=0.10, wrong_frac=0.05)),
}
DEG_CONFIGS = ("C0_baseline", "C1_tags", "C4_dist_tags_joinconv", "C5_dist_tags_joinall")


def degrade_two_axes(chunks, topical_tags, monuments=MONUMENT_TAGS, topical=None, monument=None, seed=0):
    """Degrada i tag dei chunk SEPARATAMENTE sui due assi. topical: kwargs di degrade_tags (aspetto); monument: missing_frac / wrong_frac."""
    from src.evaluation.degrade_tags import degrade_tags
    mons = set(monuments); topical, monument = dict(topical or {}), dict(monument or {})
    top_in = [{**c, "tags": [t for t in c["tags"] if t not in mons]} for c in chunks]
    top_out, log = degrade_tags(top_in, list(topical_tags), seed=seed,
                                missing_frac=topical.get("missing_frac", 0.0), wrong_frac=topical.get("wrong_frac", 0.0),
                                drop_secondary=topical.get("drop_secondary", False))
    rng = random.Random(seed + 1000)
    out = []
    for c, t in zip(chunks, top_out):
        m = [x for x in c["tags"] if x in mons]
        if m and rng.random() < monument.get("missing_frac", 0.0):
            m = []; log["monument_missing"] = log.get("monument_missing", 0) + 1
        elif m and rng.random() < monument.get("wrong_frac", 0.0):
            m = [rng.choice([x for x in monuments if x != m[0]])]; log["monument_wrong"] = log.get("monument_wrong", 0) + 1
        out.append({**c, "tags": m + t["tags"]})
    return out, log


def degradation_study(chunks, tags, base_sidecar, work_dir, make_bench, variants=None, seeds=(0, 1, 2),
                      configs=DEG_CONFIGS, prefix="deg"):
    """Per ogni variante x seed: copia del sidecar BASE (tag + predq, nessuna distanza), tag degradati sui due assi, nuovo handler/bench, retrieval.
    make_bench(sidecar_path) -> RagasBenchmark (a te la costruzione: vedi la cella del notebook).
    Distanze assenti di proposito: si isola l'effetto dei tag. tag_esperti non dipende dal seed (1 solo run)."""
    import shutil
    from src.evaluation.json_ingest import load_tags_into_sidecar
    work_dir = Path(work_dir); work_dir.mkdir(parents=True, exist_ok=True)
    topical_tags = [t for t in tags if t not in MONUMENT_TAGS]
    frames = []
    for name, (kt, km) in (variants or VARIANTS).items():
        for seed in ((0,) if name == "tag_esperti" else seeds):
            sp = work_dir / f"{prefix}_{name}_s{seed}.json"
            shutil.copy(base_sidecar, sp)
            deg, log = degrade_two_axes(list(chunks.values()), topical_tags, topical=kt, monument=km, seed=seed)
            load_tags_into_sidecar(deg, sp).release_path()
            bench = make_bench(sp)
            g = run(bench, f"{prefix}_{name}_s{seed}", configs=configs)
            frames.append(g.assign(variant=name, seed=seed))
            print(f"[{name} s{seed}] {log} | senza monumento: {sum(not set(c['tags']) & set(MONUMENT_TAGS) for c in deg)}/{len(deg)}")
    return pd.concat(frames, ignore_index=True)


def summarize_vs_c0(df, metrics=("gold_recall", "gold_mrr"), subsets=("normali", "ambigue normali")):
    """Differenza di ogni config rispetto a C0 (stessa variante e seed), media e dev.std. sui seed."""
    rows = []
    for s in subsets:
        d = df[_mask(df, s)]
        for met in metrics:
            t = d.groupby(["variant", "seed", "config"])[met].mean().unstack("config")
            diff = t.sub(t["C0_baseline"], axis=0).drop(columns="C0_baseline")
            agg = diff.groupby(level="variant").agg(["mean", "std"]).round(3)
            agg.columns = [f"{c}|{a}" for c, a in agg.columns]
            agg.insert(0, "metric", met); agg.insert(0, "subset", s)
            rows.append(agg.reset_index())
    return pd.concat(rows, ignore_index=True)


# ------------------------------------------------------------------------------------------------ 4. distanze
DIST_CONFIGS = ("C0_baseline", "C1_tags", "C3_dist_tags", "C4_dist_tags_joinconv", "C5_dist_tags_joinall")


REGIMES = ((15, 5), (10, 5), (7, 5), (5, 8), (5, 10))   # (top_k, top_j); i regimi con top_k < top_j fanno sempre partire le distanze


def distance_ablation(bench_dist, bench_nodist, regimes=REGIMES, configs=DIST_CONFIGS, prefix="dist"):
    """bench_dist: sidecar con le distanze esperte. bench_nodist: copia del sidecar BASE (stessi tag, nessuna distanza).
    Per ogni regime (top_k, top_j): effetto delle distanze = run(con) - run(senza), appaiato per query; C0 sempre a top_j chunk
    (run_regime). Restituisce (tabella, generazioni {(k, j): (senza, con)})."""
    gens, tabs = {}, []
    for k, j in regimes:
        a = run_regime(bench_nodist, f"{prefix}_nodist_k{k}_j{j}", k, j, configs=configs)
        b = run_regime(bench_dist, f"{prefix}_dist_k{k}_j{j}", k, j, configs=configs)
        gens[(k, j)] = (a, b)
        t = paired_between_runs(a, b, subsets=("normali", "ambigue normali"))
        pa = a.pivot(index="query_id", columns="config", values="retrieved_ids")
        pb = b.pivot(index="query_id", columns="config", values="retrieved_ids")
        pred = a.drop_duplicates("query_id").set_index("query_id").is_predetermined.astype(bool)
        act = {c: float(np.mean([set(x) != set(y) for x, y in zip(pa.loc[~pred, c], pb.loc[~pred, c])])) for c in pa.columns}
        t["top_k"], t["top_j"] = k, j
        t["quota_query_cambiate"] = t.config.map(act).round(2)   # normali in cui le distanze cambiano l'insieme dei top-J
        tabs.append(t)
    return pd.concat(tabs, ignore_index=True), gens


def vs_c0_by_topk(gens, side=1, metric="gold_recall", subsets=("normali", "ambigue normali")):
    """Per ogni top_k: differenza di ogni config con C0, per il sidecar CON (side=1) o SENZA (side=0) distanze."""
    rows = []
    for (k, j), pair in gens.items():
        g = pair[side]
        for s in subsets:
            d = g[_mask(g, s)]
            t = d.groupby("config")[metric].mean()
            rows += [{"top_k": k, "top_j": j, "subset": s, "config": c, f"{metric}": round(v, 3), "diff_vs_C0": round(v - t["C0_baseline"], 3)}
                     for c, v in t.items()]
    return pd.DataFrame(rows)
EOF

## ARMSTRONG DATASET

In [5]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/evaluation/json_ingest.py
"""
Ingestione di un dataset gia' chunkato (JSON) + tag dei chunk nel sidecar, in un solo passaggio.
Salvare come ~/tesi_graphrag/src/evaluation/json_ingest.py

Formato chunk: [{"chunk_id", "doc_id", "true_domain", "chunk_index", "tags": [...], "text"}]
I tag NON vanno nel payload Qdrant (convenzione del progetto): finiscono in sidecar["tag_overrides"][chunk_id]["user_tags"],
e quindi in global_tags (colori inclusi), esattamente come farebbe il pre-tagging ma senza TagAssigner.
"""
import json, sys, time, uuid
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path("~/tesi_graphrag").expanduser().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import EMBEDDING_MODEL, QDRANT_URL
from src.sidecar_manager import SidecarManager


def load_tags_into_sidecar(chunks, sidecar_path, reset_tags=True) -> SidecarManager:
    """Scrive i tag di ogni chunk nel sidecar. reset_tags=True azzera prima tag_overrides e global_tags
    (le distanze e le predq gia' presenti NON vengono toccate)."""
    sm = SidecarManager(filepath=sidecar_path)
    if reset_tags:
        data = sm.load_data(force_reload=True)
        data["tag_overrides"], data["global_tags"] = {}, {}
        sm.save_data(data)
    sm.add_tag_overrides_batch({c["chunk_id"]: list(c["tags"]) for c in chunks})
    return sm


def ingest_chunks_json(chunks_path, collection_name, sidecar_path, recreate=True, batch_size=64, embeddings=None, client=None):
    from langchain_core.documents import Document
    from langchain_community.embeddings import FastEmbedEmbeddings
    from langchain_qdrant import QdrantVectorStore
    from qdrant_client import QdrantClient
    from qdrant_client.http.models import Distance, VectorParams

    chunks = json.load(open(Path(chunks_path).expanduser(), encoding="utf-8"))
    ids = [c["chunk_id"] for c in chunks]
    assert len(ids) == len(set(ids)), "chunk_id duplicati"

    client = client or QdrantClient(url=QDRANT_URL)
    embeddings = embeddings or FastEmbedEmbeddings(model_name=EMBEDDING_MODEL)
    dim = len(embeddings.embed_query("test"))
    if recreate and client.collection_exists(collection_name):
        client.delete_collection(collection_name)
    if not client.collection_exists(collection_name):
        client.create_collection(collection_name, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))

    docs = [Document(page_content=c["text"], metadata={
        "chunk_id": c["chunk_id"], "doc_id": c["doc_id"], "shortened_doc_id": c["doc_id"],
        "true_domain": c.get("true_domain"), "chunk_index": c["chunk_index"], "user_tags": []}) for c in chunks]
    store = QdrantVectorStore(client=client, collection_name=collection_name, embedding=embeddings)
    t0 = time.perf_counter()
    for i in range(0, len(docs), batch_size):
        b = docs[i:i + batch_size]
        store.add_documents(b, ids=[str(uuid.uuid5(uuid.NAMESPACE_DNS, d.metadata["chunk_id"])) for d in b])

    reg_path = PROJECT_ROOT / "data/collection_registry.json"
    reg = json.load(open(reg_path, encoding="utf-8")) if reg_path.exists() else {}
    reg[collection_name] = {"embedding_model": EMBEDDING_MODEL, "vector_dimension": dim, "distance_metric": "COSINE",
                            "total_chunks": len(docs), "updated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
    reg_path.parent.mkdir(parents=True, exist_ok=True)
    json.dump(reg, open(reg_path, "w", encoding="utf-8"), indent=2, ensure_ascii=False)

    sm = load_tags_into_sidecar(chunks, sidecar_path)
    print(f"!>> {len(docs)} chunk in '{collection_name}' ({time.perf_counter()-t0:.1f}s); tag nel sidecar: {list(sm.get_global_tags())}")
    return sm
EOF

In [10]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/evaluation/distance_presets.py
"""
Pre-istanziazione delle classi di distanza (sidecar) per il benchmark Armstrong.
Salvare come ~/tesi_graphrag/src/evaluation/distance_presets.py

Due passi, separati apposta per poter RILEGGERE E RITOCCARE il piano prima di applicarlo:
  plan = build_plan(...)  -> dict serializzabile (salvabile in JSON, modificabile a mano)
  apply_plan(sm, plan)    -> scrive le adiacenze nel sidecar

Regole (solo sulle PREDETERMINATE; le normali non vengono mai toccate, cosi' la generalizzazione resta misurabile):
  1. predq -> chunk gold:  il primo gold = classe forte di avvicinamento, gli altri = classe lieve.
  2. predq -> chunk sbagliati (solo se si passa `bench`): i chunk NON gold che il retrieval vettoriale mette nei top_j
     della predq -> classe lieve di allontanamento; i primi 3 del ranking -> classe forte di allontanamento.
  3. chunk <-> chunk: tutte le coppie dei gold di una stessa predq -> classe lieve di avvicinamento.
     Sono le uniche modifiche che raggiungono le query normali (il 2° retrieval parte dai chunk di categoria 1).
     Con cluster_union=True (versione ESPERTA) si collega l'intero cluster tematico: gold della predq + gold della normale
     abbinata (paired_with); le coppie in cui ENTRAMBI i chunk sono gold di entrambe le query (il nucleo) usano la classe forte.
     ATTENZIONE: cosi' l'esperto "conosce" gli argomenti delle normali -> e' un tetto (upper bound), non una misura di generalizzazione.
     Le coppie chunk<->chunk di ALLONTANAMENTO non hanno effetto sul ranking (il 2° retrieval considera solo fattori > 1): non vengono generate.
Versione UTENTE MEDIO: degrade_plan(plan, keep=0.6, seed=...) tiene a caso una frazione degli stessi interventi (da scrivere in un altro sidecar).
Le classi sono scelte dalla scala attiva nel sidecar (get_class_factors): funziona anche con le classi custom di 08_01.
"""
import itertools, json
from pathlib import Path


def pick_classes(sm):
    f = sm.get_class_factors()
    up = sorted(((v, k) for k, v in f.items() if v > 1.0), reverse=True)
    down = sorted((v, k) for k, v in f.items() if v < 1.0)
    if not up:
        raise ValueError("La scala non ha classi di avvicinamento (fattore > 1).")
    return {"strong_up": up[0][1], "mild_up": up[1][1] if len(up) > 1 else up[0][1],
            "strong_down": down[0][1] if down else None, "mild_down": down[-1][1] if down else None}


def build_plan(queries, gold, sm, bench=None, top_j=None, top_k=None, penalize_wrong=True, link_gold_pairs=True, cluster_union=True):
    """queries: lista di eval_queries.json; gold: {query_id: [chunk_id,...]} con il chunk piu' importante per primo.
    bench: RagasBenchmark (opzionale) per individuare i chunk sbagliati col retrieval vettoriale reale."""
    from src.evaluation.benchmark_runner import _rec_info
    cls = pick_classes(sm)
    plan = {"classes": cls, "predq": {}, "pairs": []}
    by_id = {q["id"]: q for q in queries}
    seen_pairs = set()
    for q in queries:
        if not q.get("is_predetermined"):
            continue
        g = list(gold[q["id"]])
        edits = {c: (cls["strong_up"] if i == 0 else cls["mild_up"]) for i, c in enumerate(g)}
        if bench is not None and penalize_wrong and cls["mild_down"]:
            bench.h.set_parameters(top_j=top_j or bench.top_j, top_k=top_k or bench.top_k, top_n=bench.top_n)
            bench.h.update_query(new_text=q["query"])
            bench.h.run_vectorial_retrieval()
            ranked = [_rec_info(r, i)[0] for i, r in enumerate(bench.h.raw_records)]
            for rank, cid in enumerate(ranked[: (top_j or bench.top_j)]):
                if cid not in g:
                    edits[cid] = cls["strong_down"] if rank < 3 else cls["mild_down"]
        plan["predq"][f"predq_{q['id']}"] = edits
        if link_gold_pairs:
            partner = by_id.get(q.get("paired_with"))
            g2 = list(gold.get(partner["id"], [])) if (cluster_union and partner) else []
            core = set(g) & set(g2)
            for a, b in itertools.combinations(list(dict.fromkeys(g + g2)), 2):
                key = tuple(sorted((a, b)))
                c = cls["strong_up"] if (a in core and b in core) else cls["mild_up"]
                if key not in seen_pairs:
                    seen_pairs.add(key)
                    plan["pairs"].append([key[0], key[1], c])
                else:   # coppia gia' vista da un'altra predq: vince la classe piu' forte
                    for p in plan["pairs"]:
                        if (p[0], p[1]) == key and c == cls["strong_up"]:
                            p[2] = c
    return plan


def degrade_plan(plan, keep=0.6, seed=0, class_noise=0.0):
    """Versione "utente medio": tiene a caso una frazione `keep` degli interventi dell'esperto.
    - predq->chunk: per ogni predq si tiene round(keep*n) interventi (almeno 1), scelti a caso;
    - coppie chunk<->chunk: si tiene a caso una frazione `keep` del totale;
    - class_noise: probabilita' di scambiare classe forte/lieve (stessa direzione) in un intervento tenuto (default 0 = solo omissioni).
    Riproducibile con `seed`. Il piano originale non viene modificato."""
    import random
    rng = random.Random(seed)
    cls = plan["classes"]
    swap = {cls["strong_up"]: cls["mild_up"], cls["mild_up"]: cls["strong_up"]}
    if cls.get("strong_down") and cls.get("mild_down"):
        swap.update({cls["strong_down"]: cls["mild_down"], cls["mild_down"]: cls["strong_down"]})
    noisy = lambda c: swap.get(c, c) if rng.random() < class_noise else c
    out = {"classes": dict(cls), "predq": {}, "pairs": [],
           "meta": {"variant": "average_user", "keep": keep, "seed": seed, "class_noise": class_noise}}
    for pid, m in plan["predq"].items():
        items = list(m.items())
        k = max(1, round(keep * len(items))) if items else 0
        out["predq"][pid] = {c: noisy(v) for c, v in rng.sample(items, k)}
    k = round(keep * len(plan["pairs"]))
    out["pairs"] = [[p[0], p[1], noisy(p[2])] for p in rng.sample(plan["pairs"], k)]
    return out


def plan_stats(plan):
    from collections import Counter
    return {"predq_edits": sum(len(m) for m in plan["predq"].values()), "chunk_pairs": len(plan["pairs"]),
            "classes": dict(Counter([c for m in plan["predq"].values() for c in m.values()] + [p[2] for p in plan["pairs"]]))}


def build_expert_plan(queries, gold, sm, curation, bench=None, top_j=None, top_k=None, include_cross=True, penalize_wrong=True):
    """Piano "esperto di dominio": il grafo e' tarato su cio' che i chunk DICONO (curation.GROUPS/CROSS), non sul gold.
    - chunk<->chunk: per ogni gruppo, coppie del nucleo in classe forte, tutte le altre in classe lieve (query-indipendenti);
    - predq->chunk: gold (il primo forte, gli altri lievi) + `support` della curation (lievi) + `away` della curation;
      con `bench`, anche i chunk sbagliati che il retrieval vettoriale mette nei top_j (la curation, se presente, vince).
    Il gold serve solo per sapere quali chunk sono la risposta della predq; non genera le coppie chunk<->chunk."""
    from src.evaluation.benchmark_runner import _rec_info
    cls = pick_classes(sm)
    plan = {"classes": cls, "predq": {}, "pairs": [], "pair_notes": {}, "meta": {"variant": "expert_curated", "include_cross": include_cross}}
    rank = {"s": cls["strong_down"], "m": cls["mild_down"]}
    for q in queries:
        if not q.get("is_predetermined"):
            continue
        g = list(gold[q["id"]]); cur = curation.PREDQ.get(q["id"], {"support": [], "away": {}})
        e = {c: (cls["strong_up"] if i == 0 else cls["mild_up"]) for i, c in enumerate(g)}
        for c in cur["support"]:
            e.setdefault(c, cls["mild_up"])
        if bench is not None and penalize_wrong and cls["mild_down"]:
            bench.h.set_parameters(top_j=top_j or bench.top_j, top_k=top_k or bench.top_k, top_n=bench.top_n)
            bench.h.update_query(new_text=q["query"]); bench.h.run_vectorial_retrieval()
            ranked = [_rec_info(r, i)[0] for i, r in enumerate(bench.h.raw_records)]
            for r_, cid in enumerate(ranked[: (top_j or bench.top_j)]):
                if cid not in e:
                    e[cid] = cls["strong_down"] if r_ < 3 else cls["mild_down"]
        for c, lvl in cur["away"].items():
            if c not in g and rank.get(lvl):
                e[c] = rank[lvl]
        plan["predq"][f"predq_{q['id']}"] = e
    pairs = {}
    groups = dict(curation.GROUPS)
    if include_cross:
        groups.update(curation.CROSS)
    for name, (core, others) in groups.items():
        members = list(dict.fromkeys(core + others))
        for a, b in itertools.combinations(members, 2):
            key = tuple(sorted((a, b)))
            c = cls["strong_up"] if (a in core and b in core) else cls["mild_up"]
            if key not in pairs or (c == cls["strong_up"] and pairs[key] != cls["strong_up"]):
                pairs[key] = c
            plan["pair_notes"].setdefault("|".join(key), []).append(name)
    plan["pairs"] = [[a, b, c] for (a, b), c in sorted(pairs.items())]
    return plan


def save_plan(plan, path):
    Path(path).expanduser().write_text(json.dumps(plan, indent=2, ensure_ascii=False), encoding="utf-8")


def load_plan(path):
    return json.loads(Path(path).expanduser().read_text(encoding="utf-8"))


def reset_distance_edits(sm):
    """Azzera TUTTE le adiacenze (chunk<->chunk e predq<->chunk). Tag, predq registrate e scala delle classi restano."""
    d = sm.load_data(force_reload=True)
    d["modified_adjacencies"] = {}
    for p in d.get("predetermined_queries", {}).values():
        p["adjacencies"] = {}
    sm.save_data(d)


def apply_plan(sm, plan, reset_first=True):
    """Scrive il piano nel sidecar. Le predq devono essere gia' registrate (bench.register_predetermined())."""
    registered = set(sm.load_data(force_reload=True).get("predetermined_queries", {}))
    missing = [p for p in plan["predq"] if p not in registered]
    if missing:
        raise RuntimeError(f"Predq non registrate nel sidecar: {missing[:3]}... Esegui prima bench.register_predetermined().")
    if reset_first:
        reset_distance_edits(sm)
    edits = [{"chunk_1": pid, "chunk_2": cid, "distance_class": c} for pid, m in plan["predq"].items() for cid, c in m.items()]
    edits += [{"chunk_1": p[0], "chunk_2": p[1], "distance_class": p[2]} for p in plan["pairs"]]
    sm.save_pairwise_class_edits_batch(edits)
    return {"predq_edits": sum(len(m) for m in plan["predq"].values()), "chunk_pairs": len(plan["pairs"])}
EOF

In [11]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/evaluation/armstrong_curation.py
"""
Taratura "da esperto di dominio" del grafo Armstrong (scritta a mano leggendo TUTTI i chunk, non derivata dal gold).
Salvare come ~/tesi_graphrag/src/evaluation/armstrong_curation.py

GROUPS: gruppi di chunk che un esperto collega perche' raccontano la stessa storia/evento/periodo, INDIPENDENTEMENTE dalle query.
        "core" = nucleo del gruppo (coppie in classe FORTE); gli altri "members" si collegano a tutti in classe LIEVE.
CROSS : gruppi trasversali ai tre Armstrong (stesso tipo di informazione: onorificenze, nascita, squadra...). Disattivabili.
PREDQ : per ogni predeterminata, interventi che un esperto farebbe guardando la query:
        support = chunk non gold ma utili (avvicinati lievi); away = chunk simili ma fuorvianti (s = forte, m = lieve).
Tutto modificabile: il piano generato (JSON) resta rivedibile prima di applicarlo.
"""
L = lambda *n: [f"louis_armstrong_{i:02d}" for i in n]
C = lambda *n: [f"lance_armstrong_{i:02d}" for i in n]
N = lambda *n: [f"neil_armstrong_{i:02d}" for i in n]

# ---- storie interne a ciascun Armstrong: nome -> (core, altri membri)
GROUPS = {
 # Louis
 "L_childhood":      (L(1, 2),          L(3, 23)),          # infanzia; l'autobiografia 'My Life in New Orleans' racconta l'infanzia
 "L_apprenticeship": (L(3, 4),          L(5, 6)),           # Oliver, riverboat, Chicago, New York
 "L_20s_recordings": (L(6, 7, 8),       L(4, 5, 9)),        # Hot Five/Seven, scat, West End Blues
 "L_innovation":     (L(8, 9),          L(6, 10)),          # il solista al centro del jazz
 "L_voice":          (L(7, 10),         L(24)),             # scat e stile vocale
 "L_expansion":      (L(11, 12),        L(13, 14)),         # anni 30-50: tour, All Stars, film, diplomazia
 "L_sixties_hits":   (L(15, 16, 17),    L(21)),             # Hello Dolly, What a Wonderful World, anni 60
 "L_race_politics":  (L(18, 19),        L(14)),             # Little Rock, critiche, tour cancellato
 "L_end_legacy":     (L(20, 22),        L(21, 23, 24)),     # morte, casa-museo, premi, libri, soprannomi
 # Lance
 "C_youth":          (C(1, 2),          C(3)),
 "C_early_pro":      (C(3, 4),          C(2)),
 "C_cancer_return":  (C(5, 23, 7),      C(6, 4)),           # diagnosi, Cofidis/Postal, 1999
 "C_foundation":     (C(5, 6),          C(19)),             # fondazione Livestrong (+ dimissioni da presidente)
 "C_tour_wins":      (C(8, 22),         C(7, 9, 12, 14, 24)),
 "C_tactics_team":   (C(10, 11),        C(9, 12, 8)),       # preparazione, squadra, 'the Look', rivali
 "C_doping_arc":     (C(16, 17, 18),    C(15, 19, 20, 24, 13, 21)),
 "C_stripped":       (C(17, 13),        C(20, 16)),         # titoli e medaglia revocati
 # Neil
 "N_youth":          (N(2, 3),          N(1)),
 "N_test_pilot":     (N(4, 5),          N(3, 22, 6)),
 "N_selection":      (N(5, 6),          N(7, 4)),
 "N_gemini8":        (N(7, 8),          N(6, 9, 22)),
 "N_emergencies":    (N(8, 9),          N(11, 22)),         # gemini 8, LLRV, allarmi dell'allunaggio: sangue freddo
 "N_apollo11":       (N(11, 12, 13),    N(10, 14, 24, 23)),
 "N_apollo_prep":    (N(10, 24),        N(9, 22)),
 "N_postwar_life":   (N(16, 18),        N(17, 19)),
 "N_nasa_advocacy":  (N(17, 19),        N(16)),
 "N_death_legacy":   (N(20, 21),        N(15, 23, 18)),
 "N_honors":         (N(15, 23),        N(12)),
}

# ---- gruppi trasversali (stesso tipo di informazione nei tre Armstrong)
CROSS = {
 "X_birth":      (L(1) + C(1) + N(1),        []),
 "X_start":      (L(2) + C(2) + N(2),        L(3) + N(3)),
 "X_honors":     (L(21) + C(20) + N(15),     []),
 "X_memorials":  (L(22) + N(21, 23) + C(21), []),
 "X_health":     (C(5) + L(20) + N(20),      []),
 "X_danger":     (C(5) + N(8, 9),            []),
 "X_records":    (L(15) + C(22) + N(12, 7),  []),
 "X_teams":      (L(6, 12) + C(11) + N(10, 7), L(4) + C(10)),
 "X_sixties":    (L(17) + N(7, 12, 10),      L(15, 16) + C(1)),
}

# ---- interventi per predeterminata: support (non gold, utili) e away (simili ma fuorvianti; s = forte, m = lieve)
PREDQ = {
 "AMB_P01": {"support": L(3),        "away": {**{c: "s" for c in N(1) + C(1)}, **{c: "m" for c in N(2) + C(2)}}},
 "AMB_P02": {"support": C(22),       "away": {c: "m" for c in C(14, 3)}},
 "AMB_P03": {"support": N(9),        "away": {c: "m" for c in N(11, 5)}},
 "AMB_P04": {"support": L(18),       "away": {c: "m" for c in C(15) + N(19, 18)}},
 "AMB_P05": {"support": C(15, 19),   "away": {c: "m" for c in L(18, 19) + N(19)}},
 "AMB_P06": {"support": [],          "away": {c: "s" for c in L(20)}},
 "SPE_P01": {"support": L(4, 5),     "away": {c: "m" for c in L(12)}},
 "SPE_P02": {"support": L(17, 21),   "away": {c: "m" for c in L(16, 13)}},
 "SPE_P03": {"support": C(9),        "away": {}},
 "SPE_P04": {"support": C(7),        "away": {c: "m" for c in L(20) + N(20, 9)}},
 "SPE_P05": {"support": N(12, 10),   "away": {c: "m" for c in N(8, 9)}},
 "SPE_P06": {"support": N(4, 6),     "away": {c: "m" for c in N(3)}},
 "MUL_P01": {"support": C(17),       "away": {}},
 "MUL_P02": {"support": L(3) + N(3), "away": {}},
 "MUL_P03": {"support": [],          "away": {}},
 "MUL_P04": {"support": C(8) + N(10), "away": {}},
 "MUL_P05": {"support": L(16) + N(9, 11, 15), "away": {}},
 "MUL_P06": {"support": L(12),       "away": {}},
}
EOF

## GENERIC DATASET FROM PDF

In [1]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/evaluation/text_cleaning.py
import re
from collections import Counter

CJK = re.compile(r"[぀-ヿ一-鿿가-힯]")
REF = re.compile(r"^\s*(references|bibliography)\s*$", re.I)
CAP = re.compile(r"^\s*(figure|fig\.)\s*\d+\s*[:.]", re.I)
PAGENUM = re.compile(r"\d{1,5}")

# --- regole aggiuntive (attive solo con extended=True, vedi clean_pages) -------------------------------------------
ACK = re.compile(r"^\s*acknowledg(e)?ments?\b", re.I)                       # "Acknowledgment(s)" -> inizio della coda
PAGELABEL = re.compile(r"\d{1,3}:\d{1,3}")                                  # etichette di pagina tipo LIPIcs "35:2"
BOILERPLATE = re.compile(r"^Downloaded .*Redistribution subject to SIAM", re.I)   # riga di copyright SIAM (>120 car.)
TOC_LEADER = re.compile(r"(\.\s){5,}")                                      # righe d'indice "Introduction . . . . 223"
LICENSE = re.compile(r"(?ms)^Funding Open access.*?licenses/by/4\.0/\.[ \t]*$")   # blocco licenza Springer


def clean_pages(pages, min_chars=200, extended=False, first_page_start=None):
    """Pulisce le pagine caricate (Document LangChain): header/footer ripetuti, numeri di pagina,
    didascalie 'Figure N:', righe con CJK e coda 'References'.

    extended=False (default) = comportamento originale: la coda 'References' taglia solo il resto della
    pagina in cui compare (le pagine successive restano).
    extended=True aggiunge:
      - dopo 'References'/'Bibliography'/'Acknowledgment(s)' scarta anche TUTTE le pagine successive;
      - righe-indice con puntini di guida, etichette di pagina 'NN:NN', riga di copyright SIAM, blocco licenza Springer.
    first_page_start: regex; nella prima pagina si scartano le righe che precedono la prima riga che la soddisfa
    (serve per le pagine d'apertura dei capitoli di handbook, che iniziano con indice e affiliazioni).
    """
    cnt = Counter(l.strip() for p in pages for l in set(p.page_content.split("\n")) if l.strip())
    repeated = {l for l, n in cnt.items() if n >= 3 and len(l) < 120}
    start_re = re.compile(first_page_start) if first_page_start else None
    out, tail_reached = [], False
    for n, p in enumerate(pages):
        if tail_reached:
            continue
        text = LICENSE.sub("", p.page_content) if extended else p.page_content
        lines = text.split("\n")
        if start_re and n == 0:
            idx = next((i for i, l in enumerate(lines) if start_re.match(l.strip())), None)
            if idx is not None:
                lines = lines[idx:]
        kept, in_caption = [], False
        for line in lines:
            s = line.strip()
            if REF.match(s) or (extended and ACK.match(s)):
                tail_reached = extended
                break
            if CAP.match(s):
                in_caption = True
            if in_caption:
                if s.endswith("."):
                    in_caption = False
                continue
            if not s or s in repeated or CJK.search(s) or PAGENUM.fullmatch(s):
                continue
            if extended and (PAGELABEL.fullmatch(s) or BOILERPLATE.match(s) or TOC_LEADER.search(s)):
                continue
            kept.append(line)
        p.page_content = "\n".join(kept)
        if len(p.page_content) >= min_chars:
            out.append(p)
    return out


# --- DataSet "dst" (12 PDF: libro + handbook + paper di Tamassia, Rahman, Didimo, Bekos, ...) ---------------------
# pages: intervalli 1-based inclusivi di pagine PDF da TENERE (metadata["page"] resta quello originale, 0-based).
# first_page_start: vedi clean_pages. I documenti non elencati si puliscono con la sola clean_pages(extended=True).
DST_RULES = {
    # Libro di 400 pagine: si tengono solo le parti sull'orthogonal drawing/bend (prefazione con le applicazioni,
    # cap. 1-2, sez. 4.9, cap. 5, 7, 8 senza esercizi, Appendice A "Bounds"); fuori: copertina/indici, cap. 3, 6, 9, 10, 11,
    # resto del cap. 4, bibliografia e indice analitico.
    "Di Batista, Eades, Tamassia, Tollis - Graph Drawing": {
        "pages": [(7, 8), (12, 49), (138, 141), (145, 177), (222, 244), (246, 269), (345, 362)],
    },
    "handbook-Planar-Orthogonal-and-Polyline-Drawing-Algorithms": {"first_page_start": r"One can assess"},
    "handbook-rectangular-drawing-algorithms": {"first_page_start": r"A rectangular drawing of a plane graph"},
}


def clean_dst_pages(pages, filename):
    """Selezione pagine (se prevista) + pulizia estesa per un PDF del dataset dst. filename: nome senza '.pdf'."""
    rule = DST_RULES.get(filename, {})
    ranges = rule.get("pages")
    if ranges:
        pages = [p for p in pages if any(a <= int(p.metadata.get("page", 0)) + 1 <= b for a, b in ranges)]
    return clean_pages(pages, extended=True, first_page_start=rule.get("first_page_start"))
EOF

In [1]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/evaluation/tag_handling.py
"""
Tag handling "specifico" per il benchmark (dataset del professore o simili), SENZA modificare TagAssigner / PreTagger.

Idea: al posto dell'espansione LLM del tag (generica => tag assegnati a quasi tutti i chunk) ogni tag ha un'espansione
SCRITTA A MANO come liste di termini del dominio:
  - "strong": termini/locuzioni specifiche (peso 2)
  - "weak"  : termini piu' generici (peso 1)
Un tag e' assegnato a un testo se la somma dei pesi dei termini DISTINTI trovati e' >= MIN_SCORE (default 2: basta un
termine strong oppure due weak). Per testo si tengono al massimo MAX_TAGS tag (i piu' punteggiati): meglio qualche tag
in meno che in piu'. Nessun embedding e nessuna soglia da calibrare: il risultato e' deterministico e si puo' verificare
a mano (vedi tag_report / explain).

Uso (notebook 09_, kernel con `handler` gia' creato):
    from src.evaluation.tag_handling import DSP_TAG_SPECS, pretag, tag_report, explain
    report = pretag(handler)                  # backup + azzera i tag dei chunk + pre-tagging con le espansioni a mano
    display(report)
    explain(handler, "Didimo_SeriesParallel_chunk_9")   # perche' un chunk ha (o non ha) certi tag

`pretag` installa un assegnatore specifico in handler.tag_assigner: lo usano sia il pre-tagging sia l'assegnazione
automatica dei tag alle query (config con auto_tags=True). Tag non presenti negli specs (es. aggiunti dal widget)
ricadono sul TagAssigner originale. Per un altro dataset basta passare altri specs (stesso formato).
"""
import re
import shutil
import unicodedata
from pathlib import Path
from typing import Dict, List, Optional

from src.tag_assigner import TagAssigner

MIN_SCORE = 2     # soglia sul punteggio (strong=2, weak=1)
MAX_TAGS = 3      # massimo di tag per testo

# --------------------------------------------------------------------------------------------------------------------
# Espansioni dei 7 tag del dataset del professore (orthogonal graph drawing). Sono regex (case-insensitive) applicate
# al testo normalizzato (ligature, trattini e a-capo gia' sistemati).
# --------------------------------------------------------------------------------------------------------------------
DSP_TAG_SPECS: Dict[str, Dict[str, List[str]]] = {
    "Orthogonal Representation & Embedding": {
        "strong": [
            r"orthogonal representations?", r"angle (assignment|labell?ing)", r"planar (embedding|representation)s?",
            r"(fixed|variable|given|prescribed) embedding", r"embedding[- ]preserving", r"plane (4-)?graphs?",
            r"dimensionless", r"region[- ]preserving", r"rotation system",
        ],
        "weak": [
            r"sequence of bends", r"left and right turns?", r"outer face", r"\bfaces?\b", r"angles? (around|at) (a |each |every )?vert",
            r"\bshape\b", r"reflex", r"turn[- ]regular",
        ],
    },
    "Bend Minimization & Bounds": {
        "strong": [
            r"bend[- ]?minim\w*", r"bend[- ]optimal", r"(minimum|minimal|fewest|optimal) (number (of )?)?bends?",
            r"number of bends", r"bends? per edge", r"bend cost", r"(lower|upper) bounds? on (the )?(total |minimum |maximum )?(number of )?bends?",
            r"at most [^.]{0,40}bends", r"at least [^.]{0,40}bends",
        ],
        "weak": [
            r"\bbends?\b", r"subdivision vertices", r"dummy vert", r"(lower|upper) bounds?",
        ],
    },
    "Network Flow & Tamassia": {
        "strong": [
            r"network[- ]?flows?", r"flows?[- ]?networks?", r"min(imum)?[- ]?cost[- ]?flow", r"cost[- ]?flow", r"tamassia", r"\bTam87\b",
            r"out[- ]of[- ]kilter", r"supply and demand", r"flow model",
        ],
        "weak": [
            r"\bflows?\b", r"\bsupply\b", r"\bdemand\b", r"\barcs?\b", r"face nodes?", r"commodity", r"unit cost",
        ],
    },
    "Topology-Shape-Metrics Pipeline": {
        "strong": [
            r"topology[- ]shape[- ]metrics", r"\bTSM\b", r"shape[- ]metrics", r"shape[- ]first", r"planari[sz]ation",
            r"orthogonali[sz]ation", r"compaction", r"\bmetrics\b",
        ],
        "weak": [
            r"crossing minimi[sz]ation", r"minimi[sz]\w* (the )?(number of )?crossings", r"orthogonal crossings?",
            r"dummy vert\w* [^.]{0,40}crossings?", r"coordinates", r"edge crossings?",
        ],
    },
    "Rectilinear Planarity & Complexity": {
        "strong": [
            r"rectilinear (planar\w*|drawings?|graphs?)", r"HV[- ](rectilinear|restricted|drawings?|graphs?)",
            r"NP[- ](hard\w*|complete\w*)", r"fixed[- ]parameter", r"\bFPT\b", r"rectilinear planarity",
            r"(polynomial|linear)[- ]time (algorithm|solvable)", r"longstanding", r"open (problem|question)",
        ],
        "weak": [
            r"complexity", r"polynomial", r"linear[- ]time", r"\bSAT\b", r"reduction", r"4-planar", r"4-graphs?", r"degree[- ](four|4)",
        ],
    },
    "Series-Parallel & Cubic Graphs": {
        "strong": [
            r"series[- ]?parallel", r"\bSPQ\*?", r"spirality", r"\bcubic\b", r"tri-?connected", r"3-connected",
            r"rectangular drawings?", r"(bad|legged|maximal) cycles?", r"thomassen",
        ],
        "weak": [
            r"decomposition tree", r"\bpoles?\b", r"bi-?connected", r"[SPQR]\*?-nodes?", r"3-graphs?", r"degree[- ](three|3)\b",
        ],
    },
    "Applications & Aesthetics": {
        "strong": [
            r"aesthetics?", r"readability", r"\bVLSI\b", r"floor[- ]?plans?", r"entity[- ]relationship", r"data[- ]?flow diagrams?",
            r"organi[sz]ation charts?", r"circuit (layout|schematics?)", r"machine[- ]learning", r"three[- ]dimensional", r"\b3D\b",
            r"applications? (of|in|to|include)", r"software engineering",
        ],
        "weak": [
            r"angular resolution", r"geometric uniformity", r"\barea\b", r"transportation", r"\bdatabases?\b", r"diagrams?",
        ],
    },
}


# --------------------------------------------------------------------------------------------------------------------
def normalize_text(text: str) -> str:
    """Ligature (fl, fi), trattini unicode, parole spezzate a fine riga e spazi: cosi' le regex non perdono match."""
    t = unicodedata.normalize("NFKC", text or "")
    t = re.sub(r"[‐‑‒–—−]", "-", t)
    t = re.sub(r"-\s*\n\s*", "-", t)
    return re.sub(r"\s+", " ", t)


def compile_specs(specs: Dict[str, Dict[str, List[str]]]):
    return {tag: [(2, re.compile(p, re.I)) for p in s.get("strong", [])] + [(1, re.compile(p, re.I)) for p in s.get("weak", [])]
            for tag, s in specs.items()}


def score_text(text: str, compiled) -> Dict[str, int]:
    """Punteggio di ogni tag degli specs per il testo (somma dei pesi dei termini distinti trovati)."""
    t = normalize_text(text)
    return {tag: sum(w for w, rx in pats if rx.search(t)) for tag, pats in compiled.items()}


class SpecificTagAssigner(TagAssigner):
    """
    TagAssigner con le espansioni a mano.
    Fa il controllo dei termini presenti nel testo del chunk, e somma il punteggio relativo per i termini presenti allo score finale.
    De il punteggio di termini trovati nel chunk supera la soglia assegna il tag di cui ha preso la lista. Può assegnarne fino a un max di 3.
    Così facendo non dipende più dalla casualità dell'LLM.
    """

    @classmethod
    def from_assigner(cls, base: TagAssigner, specs=None, min_score: int = MIN_SCORE, max_tags: int = MAX_TAGS):
        new = cls.__new__(cls)
        new.__dict__.update(base.__dict__)
        new.specs = specs or DSP_TAG_SPECS
        new.compiled = compile_specs(new.specs)
        new.min_score, new.max_tags = min_score, max_tags
        return new

    def assign_tags(self, text, candidate_tags, vector=None) -> List[str]:
        known = [t for t in candidate_tags if t in self.compiled]
        scores = score_text(text, {t: self.compiled[t] for t in known})
        # a parita' di punteggio vale l'ordine dello spec; si tengono i migliori max_tags
        order = {t: i for i, t in enumerate(self.compiled)}
        ranked = sorted((t for t in known if scores[t] >= self.min_score), key=lambda t: (-scores[t], order[t]))
        assigned = ranked[: self.max_tags]
        unknown = [t for t in candidate_tags if t not in self.compiled]
        if unknown:   # tag non negli specs (es. aggiunti dal widget): comportamento originale
            assigned += super().assign_tags(text, unknown, vector)
        return assigned


def install(handler, specs=None, min_score: int = MIN_SCORE, max_tags: int = MAX_TAGS) -> SpecificTagAssigner:
    """Sostituisce handler.tag_assigner (usato da pretag_sidecar e dall'auto-tagging delle query)."""
    handler.tag_assigner = SpecificTagAssigner.from_assigner(handler.tag_assigner, specs, min_score, max_tags)
    return handler.tag_assigner


def pretag(handler, specs=None, batch_size: int = 100, min_score: int = MIN_SCORE, max_tags: int = MAX_TAGS,
           reset: bool = True):
    """Installa l'assegnatore specifico e fa il pre-tagging di tutta la collezione con PreTagger (invariato).
    reset=True: copia di backup del sidecar (<nome>_pre_tag_handling.json) e azzeramento di tag_overrides/global_tags
    (il pre-tagging aggiunge soltanto); modified_adjacencies e predetermined_queries NON si toccano.
    ATTENZIONE: i tag dei chunk modificati a mano prima di questa chiamata vanno persi (restano nel backup)."""
    specs = specs or DSP_TAG_SPECS
    install(handler, specs, min_score, max_tags)
    if reset:
        p = Path(handler.sidecar_path)
        if p.exists():
            shutil.copy(p, p.with_name(p.stem + "_pre_tag_handling" + p.suffix))
        data = handler.sidecar_manager.load_data()
        data["tag_overrides"], data["global_tags"] = {}, {}
        handler.sidecar_manager.save_data(data)
    handler.pretag_sidecar(candidate_tags=list(specs), batch_size=batch_size)
    return tag_report(handler)


def tag_report(handler):
    """Tabella: per ogni tag quanti chunk (e quale %) lo hanno, piu' chunk senza tag e tag medi per chunk."""
    import pandas as pd
    from src.config import COLLECTION_NAME
    data = handler.sidecar_manager.load_data()
    total = handler.qdrant_client.count(collection_name=COLLECTION_NAME).count
    counts = {t: v.get("count", 0) for t, v in data.get("global_tags", {}).items()}
    per_chunk = [len(v.get("user_tags", [])) for v in data.get("tag_overrides", {}).values()]
    rows = [{"tag": t, "chunk": n, "%": round(100 * n / max(total, 1), 1)} for t, n in sorted(counts.items(), key=lambda x: -x[1])]
    rows.append({"tag": "(chunk senza tag)", "chunk": total - sum(1 for n in per_chunk if n), "%": round(100 * (total - sum(1 for n in per_chunk if n)) / max(total, 1), 1)})
    rows.append({"tag": "(tag medi per chunk)", "chunk": round(sum(per_chunk) / max(total, 1), 2), "%": None})
    return pd.DataFrame(rows)


def explain(handler, chunk_id: str):
    """Punteggi per tag di un chunk, con i termini che hanno fatto match (per capire perche' un tag c'e' o manca)."""
    from src.config import COLLECTION_NAME
    recs, off = [], None
    while not recs:      # scansione in Python (2-3k chunk): evita dipendenze da indici/filtri Qdrant
        page, off = handler.qdrant_client.scroll(collection_name=COLLECTION_NAME, limit=256, offset=off, with_payload=True)
        for r in page:
            pl = r.payload or {}
            m = pl.get("metadata") if isinstance(pl.get("metadata"), dict) else pl
            if (m.get("chunk_id") or pl.get("chunk_id")) == chunk_id:
                recs = [r]; break
        if off is None: break
    if not recs:
        print("chunk non trovato:", chunk_id); return
    p = recs[0].payload or {}
    text = p.get("page_content") or p.get("text") or ""
    ta = handler.tag_assigner
    t = normalize_text(text)
    print(text[:300].replace("\n", " "), "...\n")
    for tag, pats in ta.compiled.items():
        hits = [rx.pattern for w, rx in pats if rx.search(t)]
        sc = sum(w for w, rx in pats if rx.search(t))
        print(f"{'*' if sc >= ta.min_score else ' '} {sc:2d}  {tag:40s} {hits[:5]}")
    print("\nassegnati:", ta.assign_tags(text, list(ta.compiled)))

EOF